# 03 — Nấc 3: SFT trên Qwen3-8B

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/vinumqa-numerical-reasoning/blob/main/notebooks/03_sft_qwen3.ipynb)

**Cần GPU.** Phần A ~40–60 phút, phần B ~1.5–2 giờ trên L4.

## Vì sao KHÔNG fine-tune thẳng trên nhãn vàng

Lần chạy trước đã thử đúng cách đó (Stage 2: LoRA Mistral-7B trên gold thô) và **thất bại**:

| Step | Train loss | Val loss |
|---|---|---|
| 50 | 0.7277 | 0.9122 |
| 150 | 0.3678 | 0.9268 |
| 250 | 0.2384 | 0.9473 |

Train loss giảm đều, val loss tăng rồi đứng — overfit kinh điển. Có hai nguyên nhân:
nhãn vàng có 5–10 % không nhất quán, và **đích huấn luyện chỉ là program trần, không có
chuỗi suy luận**, nên model không học được cách suy luận mà chỉ học thuộc.

## Cách làm ở nấc này: rejection sampling (self-distillation)

1. Chạy chính Qwen3-8B (chưa fine-tune) với **prompt của nấc 2** trên tập train.
2. Giữ lại những mẫu model làm **đúng** (PA hoặc EA).
3. Dùng chính lời giải đó — *đã có đầy đủ chuỗi suy luận tiếng Việt* — làm đích huấn luyện.

Ba ưu điểm so với cách SFT thẳng trên gold:

* Đích huấn luyện **có chuỗi suy luận**, đúng văn phong model vốn sinh ra được → tránh
  đúng nguyên nhân overfit đã nêu.
* **Không cần API ngoài.** Cách cũ phải dùng Gemini-2.5-flash sửa 600 mẫu, tức là rơi sang
  thiết lập *unconstrained*. Cách này vẫn nằm trong **constrained-resource**.
* Mẫu có nhãn nhiễu phần lớn tự rơi ra: model làm "đúng theo gold" rất khó khi gold sai.
  Ta còn lọc thêm bằng `data.is_noisy_gold()`.

**Hạn chế phải nêu khi báo cáo:** chỉ học được từ những gì model *đã* làm được, nên khó dạy
kiểu bài model chưa bao giờ giải đúng. Tuỳ chọn `ADD_GOLD_FALLBACK` bù một phần nhưng bật
nó là quay lại đúng rủi ro overfit của cách cũ — mặc định tắt.

## Cấu trúc notebook

* **Phần A** (§1–§5): dựng dữ liệu SFT. Cần vLLM.
* **⚠ RESTART RUNTIME** giữa hai phần — vLLM giữ VRAM rất chặt, không nhả đủ cho training.
* **Phần B** (§6–§10): huấn luyện LoRA, rồi chấm trên test.

## Tham số LoRA

Giữ nguyên `reference/original_notebooks/finetune_phi4.ipynb`: r=16, alpha=32, dropout=0,
7 target modules, `use_gradient_checkpointing="unsloth"`, `paged_adamw_8bit`,
lr=2e-4, cosine, warmup 0.1, weight_decay 0.05, early stopping patience 3.
Batch được tự chỉnh theo VRAM nhưng **batch hiệu dụng giữ nguyên 16** để lr 2e-4 còn hợp lệ.

# PHẦN A — Dựng dữ liệu SFT

## §1. Môi trường

In [ ]:
# Cài đặt — ghim theo bộ ĐÃ XÁC MINH cài xong sạch trên image Colab hiện tại
# (Python 3.13, torch 2.11.0+cu128, A100).
#
# ⚠ KHÁC bản tham chiếu, và đây là chủ ý:
#   Khối cài đặt gốc ghim transformers==4.56.2 / trl==0.22.2 / xformers==0.0.29.post3.
#   Trên image Colab hiện tại nó THẤT BẠI — nhánh chọn xformers chỉ biết torch 2.8/2.9,
#   gặp torch 2.11 thì rơi vào bản 0.0.29.post3 (dành cho torch 2.5) nên đổ cả khối,
#   mà `%%capture` lại nuốt mất báo lỗi.
#   Bộ dưới đây là bộ pip tự giải ra khi để `unsloth` và `vllm` thoả thuận với nhau.
#   Chênh lệch phiên bản được ghi vào `env` của meta mỗi nấc, nên báo cáo vẫn truy được.
#
# ⏱ 6–12 phút (đã ghim nên pip khỏi dò tìm). Cố ý KHÔNG giấu output để thấy nó còn sống.
import os, time
_t_cai = time.time()
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install unsloth==2026.9.4 transformers==4.57.6 trl==0.24.0 peft==0.20.0 bitsandbytes==0.50.2 xformers==0.0.35
    # vLLM phải khớp CUDA của torch. Bản trên PyPI dựng cho CUDA 13, còn Colab đang
    # CUDA 12.8 → unsloth CHẶN import và báo "No module named 'vllm'" dù gói vẫn có.
    # Wheel dưới đây là bản cu129, đúng cái unsloth khuyến nghị cho hệ CUDA 12.x.
    # Nếu image Colab đổi CUDA: chạy ô này, đọc dòng WARNING của unsloth ở cell sau —
    # nó in ra đúng URL wheel cần dùng, thay vào đây là xong.
    !pip install https://github.com/vllm-project/vllm/releases/download/v0.23.0/vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl
print(f"\n[CÀI ĐẶT] xong sau {(time.time() - _t_cai) / 60:.1f} phút")
print("[CÀI ĐẶT] Colab hiện nút RESTART SESSION thì bấm, rồi chạy lại TỪ CELL #2 "
      "(bỏ qua ô này — cài lại là thừa).")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, sft, stats
from vinumqa.prompts import PromptKit

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

# Thang prompt LỒNG NHAU: basic ⊂ no_fewshot ⊂ engineered — mỗi nấc thêm đúng một khối
# cắt ra từ prompt hoàn chỉnh, phần chung giống nhau từng ký tự.
LADDER = [
    ("01_basic",           "Nấc 1 — prompt cơ bản (danh sách phép toán + yêu cầu)"),
    ("02_prompt_eng",      "Nấc 2 — prompt hoàn chỉnh (+ hướng dẫn từ khoá + few-shot)"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("04_selfeval_sft",    "Nấc 4b — + self-eval (model SFT)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05_ace_sft",         "Nấc 5b — + ACE (model SFT)"),
    ("05c_ace_basic_base", "Nấc 5c — ACE trên prompt cơ bản"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("06_comb_F_A",        "Tổ hợp — SFT + ACE (không self-eval)"),
    ("08_tu_nhat_quan",    "Mới — self-consistency K mẫu (ví dụ cố định)"),
    ("09_vidu_dong",       "Mới — self-consistency + ví dụ truy hồi"),
    ("06_comb_E_A_moi",    "Mục tiêu — prompt + ACE + phương pháp mới"),
    ("06_comb_F_A_moi",    "Mục tiêu — SFT + ACE + phương pháp mới"),
]
LADDER_LABEL = dict(LADDER)

# Những biến phải đi kèm kết quả thì mới truy lại được về sau.
_CFG_KEYS = ("MODEL_NAME", "MODEL_TAG", "TEMPERATURE", "MAX_TOKENS", "REPETITION_PENALTY",
             "MAX_SEQ_LENGTH", "BATCH_SIZE", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "RANDOM_SEED")


def run_env():
    """Môi trường THẬT lúc chạy: commit, GPU, phiên bản thư viện.

    Chỉ đọc thư viện đã nạp (``sys.modules``) chứ không import thêm — vừa nhanh,
    vừa báo đúng những gì thật sự được dùng.
    """
    env = {"python": sys.version.split()[0]}
    try:                                   # bản code nào sinh ra kết quả này
        def _g(*a):
            return subprocess.run(["git", "-C", REPO_DIR, *a],
                                  capture_output=True, text=True).stdout.strip()
        env["commit"] = _g("rev-parse", "--short", "HEAD")
        env["branch"] = _g("rev-parse", "--abbrev-ref", "HEAD")
        env["dirty"] = bool(_g("status", "--porcelain"))
    except Exception:
        pass
    _torch = sys.modules.get("torch")
    if _torch is not None:
        env["torch"] = getattr(_torch, "__version__", "?")
        try:
            if _torch.cuda.is_available():
                _p = _torch.cuda.get_device_properties(0)
                env["gpu"] = _p.name
                env["vram_gb"] = round(_p.total_memory / 1024**3, 1)
                env["cc"] = f"{_p.major}.{_p.minor}"
            else:
                env["gpu"] = "CPU"
        except Exception:
            pass
    else:
        env["gpu"] = "CPU (không nạp torch)"
    for _lib in ("transformers", "trl", "peft", "vllm", "unsloth",
                 "sentence_transformers", "numpy"):
        _m = sys.modules.get(_lib)
        if _m is not None and hasattr(_m, "__version__"):
            env[_lib] = _m.__version__
    return env


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "meta":  f"{stage}_meta.json"}[kind])


# Cấu hình CHUẨN của cả thang bậc — đo trên A100 40GB.
# MAX_SEQ_LENGTH đổi theo GPU (A100 15000 / L4 13500 / T4 8192), mà đổi GPU là đổi
# thành phần lô, đổi kernel, đổi luôn token được lấy mẫu ở temperature 0.1. Hai lần
# chạy khác max_seq KHÔNG so thẳng được, nên phải ghi sang tên nấc khác.
MAX_TOKENS_CHUAN, MAX_SEQ_CHUAN = 4096, 17000



def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: jsonl + meta, kèm một file output thô.

    Chạy với ``MAX_TOKENS`` khác mức chuẩn thì tự ghi sang tên nấc khác. Đổi trần sinh
    là đổi cấu hình, kết quả không so thẳng với thang bậc được — mà nếu cứ ghi đè lên
    tên cũ thì mất luôn bản chuẩn, không lấy lại được nếu không chạy lại GPU.
    """
    _hau_to = ""
    _mt, _ms = globals().get("MAX_TOKENS"), globals().get("MAX_SEQ_LENGTH")
    if _mt and _mt != MAX_TOKENS_CHUAN:
        _hau_to += f"_tok{_mt}"
    if _ms and _ms != MAX_SEQ_CHUAN:
        _hau_to += f"_seq{_ms}"
    if _hau_to and not stage.endswith(_hau_to):
        stage = f"{stage}{_hau_to}"
        if not quiet:
            print(f"[GHI] ⚠ cấu hình khác chuẩn (max_tokens={_mt}, max_seq={_ms}, "
                  f"GPU={globals().get('_GPU', '?')}) → ghi sang nấc '{stage}'.")
            print( "       Kết quả khác GPU/khác trần KHÔNG so thẳng với thang bậc chuẩn.")
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            "enable_thinking": getattr(globals().get("prompt_kit", None),
                                       "enable_thinking", "?"),
            "ty_le_bi_cat_token": (round(ty_le_bi_cat(), 4)
                                   if "ty_le_bi_cat" in globals() else None),
            "bi_cat_theo_buoc": (bi_cat_theo_buoc()
                                if "bi_cat_theo_buoc" in globals() else None),
            "nap_an_toan": bool(globals().get("NAP_AN_TOAN", False)),
            "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
            "env": run_env(),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'meta')}")
    return stage                      # tên THẬT, có thể khác tên truyền vào


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else "máy cá nhân"
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

In [ ]:
# ═══ Self-test: chạy TRƯỚC khi tốn GPU ═══
# Cell này đỏ thì dừng lại — mọi con số PA/EA sau đó sẽ vô nghĩa.

# (1) Ô cài đặt có thật sự cài được không. Đọc metadata nên nhanh, không phải import.
#     Kiểm ở đây để lỗi pip lộ ra trong 1 giây, thay vì 20 phút nữa lúc nạp model.
from importlib.metadata import version as _ver, PackageNotFoundError as _NoPkg
_goi = {}
for _p in ("vllm", "unsloth", "transformers", "trl", "peft", "torch"):
    try:
        _goi[_p] = _ver(_p)
    except _NoPkg:
        _goi[_p] = None
print("[GÓI] " + " | ".join(f"{k}={v}" for k, v in _goi.items() if v))
# vLLM phải khớp CUDA của torch, nếu không unsloth CHẶN import dù gói vẫn có mặt —
# lúc đó cell nạp model báo "No module named 'vllm'" một cách khó hiểu.
# Wheel khớp CUDA có đuôi "+cuXXX" trong số phiên bản; bản PyPI thì không.
if _goi.get("vllm") and "+cu" not in _goi["vllm"]:
    print("[GÓI] ⚠ vllm=" + _goi["vllm"] + " là bản PyPI (dựng cho CUDA 13). Nếu cell nạp "
          "model báo \"No module named 'vllm'\" thì cài lại bằng wheel khớp CUDA — "
          "dòng WARNING của unsloth in sẵn URL đúng.")

_thieu = [k for k, v in _goi.items() if v is None]
if _thieu:
    raise RuntimeError(
        "Thiếu gói: " + ", ".join(_thieu) + " — ô cài đặt (cell #1) đã thất bại.\n\n"
        "Cách chữa: mở Cửa sổ dòng lệnh (góc dưới trái), chạy\n"
        "    pip install -U unsloth vllm\n"
        "xem lỗi thật, xong Restart session rồi chạy lại TỪ CELL #2 (bỏ qua cell #1).")

# vLLM 0.23 cấm toàn bộ transformers 5.x. Gói nào đó nâng lên 5 thì chặn ngay tại đây,
# đừng để phát hiện sau 4 phút nạp model. (sentence-transformers ≥ 6 là thủ phạm hay gặp.)
if str(_goi["transformers"]).split(".")[0] != "4":
    raise RuntimeError(
        "transformers=" + str(_goi["transformers"]) + " — vLLM 0.23 chỉ chạy với "
        "transformers 4.x, gói nào đó đã nâng nó lên.\n"
        "Chữa: pip install \"transformers==4.57.6\" rồi Restart session.")

# (2) Executor có tái tạo đúng nhãn vàng không.
_ok = sum(dsl.check_ea(dsl.execute_program(s["qa"]["program"], s.get("table") or []),
                       s["qa"].get("exe_ans")) for s in test_all)
print(f"[SELF-TEST] executor tái tạo exe_ans trên test: {_ok}/{len(test_all)}")
assert _ok / len(test_all) > 0.99, "Executor không tái tạo được nhãn vàng — DỪNG."
assert dsl.execute_program("divide(5310, add(1, 0.15))", []) is None   # lồng nhau
assert dsl.check_pa("add(1, 2)", "add(2, 1)")[0]                       # giao hoán
assert dsl.check_ea(0.6066481994, "0.60665")                           # làm tròn 5 chữ số
print("[SELF-TEST] ✅ executor / PA / EA đạt")

## §2. Model (bản gốc, chưa fine-tune)

In [ ]:
# ═══════════════ MODEL — Qwen3-8B 4-bit ═══════════════
# Tham số lấy từ reference/original_notebooks/inference_with_difference_models.ipynb:
#   load_in_4bit=True, fast_inference=True, temperature=0.1
# max_tokens thì KHÔNG giữ: nâng 3000 → 8192 vì ở mức cũ 5–10 % mẫu bị cắt giữa lúc
# suy nghĩ, mất trắng. Xem lý do đầy đủ ở ô cấu hình GPU.
MODEL_NAME = "unsloth/Qwen3-8B"
MODEL_TAG  = "Qwen3-8B"

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Không thấy GPU. Runtime → Change runtime type → L4 GPU.")
_GPU, _VRAM = torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1024**3
_CC = torch.cuda.get_device_capability(0)
if _CC[0] < 7:
    raise RuntimeError(f"{_GPU} (CC {_CC[0]}.{_CC[1]}) không chạy được vLLM. "
                       f"Runtime → Change runtime type → A100 GPU.")

# ═══ Tham số ẢNH HƯỞNG KẾT QUẢ — CỐ ĐỊNH trên mọi GPU ═══
# Trước đây max_seq đổi theo GPU (A100 15000 / L4 13500) nên hai máy cho kết quả
# không so thẳng được. Giờ khoá cứng: đổi GPU chỉ đổi tốc độ, không đổi đầu vào.
#
# TRẦN SINH = 4096. Đây là mức ĐO ĐƯỢC là tối ưu, không phải chọn bừa:
#     nấc 2, cùng prompt, cùng GPU, chỉ khác trần —
#       4096 → 30 lượt bị cắt | 28 mẫu mất | EA 0.6479 | 300 mẫu đúng
#       8192 → 30 lượt        | 28 mẫu     | EA 0.6479 | 300 mẫu đúng
#     Gấp đôi ngân sách cứu ĐÚNG 0 mẫu. Số mẫu vượt trần không phụ thuộc trần, nên 4096
#     đã qua điểm bão hoà; 8192 chỉ tốn thêm thời gian. (Dưới 4096 thì mất thêm mẫu.)
#
# ~6 % mẫu vẫn chạm trần — nay KHÔNG bỏ mặc nữa: run_pipeline vớt chúng bằng một lượt
# sinh lại với suy nghĩ TẮT (xem `vot_mau_bi_cat`). Đó mới là cách chữa, không phải trần.
#
# max_seq 17000 theo ngân sách (neo vào phép đo thật bằng tokenizer):
#     prompt bước 2 xấu nhất = 7464 + 4096 = 11560
#     ngân sách              = 17000 − 4096 = 12904   → dư 1344 token
# Ô §3 đo lại bằng tokenizer thật và tự cắt ngữ cảnh + báo động nếu tính sai.
#
# ⚠ ĐỪNG nâng tiếp. Đã có phép so SẠCH: nấc 2 chạy hai lần với CÙNG prompt engineered,
# cùng model, cùng GPU, chỉ khác trần token —
#     trần 4096 → 30 lượt sinh bị cắt | 28 mẫu mất trắng | EA 0.6479 | 300 mẫu đúng
#     trần 8192 → 30 lượt             | 28 mẫu           | EA 0.6479 | 300 mẫu đúng
# Gấp đôi ngân sách cứu được ĐÚNG 0 mẫu, đổi lại ~50 % thời gian (10,9 → 16,4 phút).
#
# Số mẫu vượt ngân sách KHÔNG phụ thuộc ngân sách → những lượt đó thực tế không có điểm
# dừng. Mà chúng cũng không lặp (§4 đo trung vị lặp = 0.0 ở nấc 2), nên repetition_penalty
# cũng không phải thuốc. Coi đây là sàn ~6 %, đều ở mọi nấc: ghi nhận rồi bỏ qua.
TEMPERATURE, MAX_TOKENS = 0.1, 4096
MAX_SEQ_LENGTH = 17000
REPETITION_PENALTY = 1.0

# ═══ Tham số chỉ ảnh hưởng TỐC ĐỘ — chỉnh theo VRAM ═══
if _VRAM < 20:
    raise RuntimeError(
        f"{_GPU} chỉ {_VRAM:.0f} GB — không đủ cho max_seq={MAX_SEQ_LENGTH}.\n"
        f"Hạ max_seq xuống thì kết quả KHÔNG so được với các nấc khác, nên thà dừng "
        f"còn hơn ra một con số không dùng được. Đổi sang L4 hoặc A100.")
# util giữ 0.85 (hạ từ 0.88 sau một lần vLLM không dựng nổi engine vì VRAM còn sót).
# MAX_NUM_SEQS trả về mức cũ được vì max_seq đã từ 25000 xuống 17000, áp lực KV giảm hẳn.
#
# BATCH_SIZE = 512 để 497 mẫu vào ĐÚNG MỘT LÔ. Đo từ log thật: lô 400 mẫu chạy
# 1,88 s/mẫu, lô 97 mẫu còn lại chạy 2,83 s/mẫu — chậm hơn 50 % vì không lấp đầy GPU mà
# vẫn phải đợi mẫu dài nhất. Gộp một lô tiết kiệm ~1,5 phút MỖI lượt sinh; nấc 4 và nấc 5
# có nhiều lượt nên cộng lại đáng kể.
elif _VRAM < 30:                           # L4 24GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.86, 16, 512
elif _VRAM < 60:                           # A100 40GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.85, 48, 512
else:                                      # A100 80GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.85, 128, 512
DTYPE = torch.float16 if _CC[0] < 8 else None

print(f"[GPU] {_GPU} | {_VRAM:.1f} GB | CC {_CC[0]}.{_CC[1]}")
print(f"[CFG] max_seq={MAX_SEQ_LENGTH} max_tokens={MAX_TOKENS} temp={TEMPERATURE} "
      f"(cố định mọi GPU) | batch={BATCH_SIZE} max_num_seqs={MAX_NUM_SEQS} (theo VRAM)")

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import unsloth
from unsloth import FastLanguageModel
from vllm import SamplingParams

torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)

import shutil as _sh

# Đặt True nếu model tải về bị thiếu trọng số: tắt hf_transfer thì tải chậm hơn vài phút
# nhưng có kiểm tra và tải tiếp được. Lưu ý: `export` trong Cửa sổ dòng lệnh KHÔNG tới
# được kernel notebook — phải đặt ở đây.
TAI_CHAM_CHO_CHAC = False
if TAI_CHAM_CHO_CHAC:
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    print("[MODEL] đã tắt hf_transfer — tải chậm hơn nhưng chắc hơn")

_free = _sh.disk_usage("/").free / 1024**3
_t_nap = time.time()
print(f"[MODEL] Đang tải {MODEL_NAME} ... (đĩa trống {_free:.0f} GB)")
if _free < 15:
    print("[MODEL] ⚠ dưới 15 GB trống — model ~15 GB, tải dễ đứt giữa chừng.")
print("[MODEL] ⏳ Mất 4–7 PHÚT. Tải xong rồi vLLM còn dựng CUDA graph — đoạn đó")
print("[MODEL]    KHÔNG có thanh tiến trình, nhìn như treo nhưng không phải.")
print("[MODEL]    Muốn biết còn sống: xem MỐC GIỜ ở các dòng INFO bên dưới. Nó nhích")
print("[MODEL]    lên là đang chạy. Đứng im quá 10 phút mới đáng nghi.")

# enable_prefix_caching: system prompt (~1 800 token) GIỐNG HỆT ở cả 497 request, nên
# vLLM chỉ cần prefill nó một lần rồi dùng lại. Tiết kiệm phần lớn thời gian prefill.
# Không đổi token sinh ra — mỗi request vẫn có seed riêng.
_NAP_KW = dict(model_name     = MODEL_NAME,
               dtype          = DTYPE,
               max_seq_length = MAX_SEQ_LENGTH,
               load_in_4bit   = True,
               fast_inference = True)
try:                                  # bản unsloth cũ không nhận tham số này
    import inspect as _insp
    if "enable_prefix_caching" in _insp.signature(
            FastLanguageModel.from_pretrained).parameters:
        _NAP_KW["enable_prefix_caching"] = True
except Exception:                                    # noqa: BLE001
    pass
NAP_AN_TOAN = False          # True = đã phải lùi về chế độ an toàn, có ghi vào meta

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        **_NAP_KW, gpu_memory_utilization=GPU_MEM_UTIL, max_num_seqs=MAX_NUM_SEQS)
except (RuntimeError, ValueError) as _e:
    # ── vLLM dựng engine hỏng vì CUDA ──
    # KHÔNG phải tải model hỏng: model đã nằm trên đĩa rồi. Lỗi ở bước cấp phát KV cache
    # và dựng CUDA graph — thường do VRAM trống ít hơn lần trước (GPU khác, hoặc tiến
    # trình cũ còn giữ bộ nhớ), khiến số block KV tính ra quá nhỏ.
    if "CUDA error" in str(_e) or "invalid argument" in str(_e):
        print("[MODEL] ⚠ vLLM KHÔNG dựng được engine (CUDA error).")
        print(f"[MODEL]   Đang dùng: max_seq={MAX_SEQ_LENGTH} util={GPU_MEM_UTIL} "
              f"max_num_seqs={MAX_NUM_SEQS}")
        try:
            _free, _tot = torch.cuda.mem_get_info()
            print(f"[MODEL]   VRAM trống: {_free/1024**3:.1f}/{_tot/1024**3:.1f} GB"
                  + ("   ← ĐÃ BỊ CHIẾM. Restart session rồi chạy lại TỪ Ô #2."
                     if _free / _tot < 0.9 else ""))
        except Exception:                                    # noqa: BLE001
            pass
        print("[MODEL]   Thử lại ở CHẾ ĐỘ AN TOÀN: bỏ CUDA graph, hạ VRAM và số chuỗi.")
        print("[MODEL]   Ba thứ đó chỉ đổi TỐC ĐỘ — mỗi request đã có seed riêng nên")
        print("[MODEL]   thành phần lô không ảnh hưởng token sinh ra.")
        gc.collect()
        torch.cuda.empty_cache()
        _an = dict(gpu_memory_utilization=min(GPU_MEM_UTIL, 0.80),
                   max_num_seqs=max(8, MAX_NUM_SEQS // 4))
        try:
            model, tokenizer = FastLanguageModel.from_pretrained(
                **_NAP_KW, enforce_eager=True, **_an)
        except TypeError:                 # bản unsloth không nhận enforce_eager
            model, tokenizer = FastLanguageModel.from_pretrained(**_NAP_KW, **_an)
        GPU_MEM_UTIL = _an["gpu_memory_utilization"]
        MAX_NUM_SEQS = _an["max_num_seqs"]
        NAP_AN_TOAN = True
        print(f"[MODEL] ✅ nạp được ở chế độ an toàn (util={GPU_MEM_UTIL} "
              f"max_num_seqs={MAX_NUM_SEQS}) — chậm hơn, kết quả không đổi.")
    # ── Thiếu trọng số: shard safetensors tải dở còn trong cache ──
    elif "not initialized from checkpoint" in str(_e):
        raise RuntimeError(
            "Model thiếu trọng số — bản tải dở trong cache HuggingFace.\n\n"
            "Bước 1 — xoá cache. Mở Cửa sổ dòng lệnh (góc dưới trái):\n"
            "    rm -rf ~/.cache/huggingface/hub/models--unsloth--Qwen3-8B*\n"
            "    df -h / | tail -1          # kiểm luôn, cần ≥ 20 GB trống\n\n"
            "Bước 2 — đặt TAI_CHAM_CHO_CHAC = True ở ĐẦU CHÍNH Ô NÀY.\n"
            "    (`export` trong terminal không tới được kernel notebook.)\n\n"
            "Bước 3 — Restart session, chạy lại TỪ CELL #2 (bỏ qua ô cài đặt).\n\n"
            "Hỏng y hệt lần nữa thì không phải do tải: khi đó là bản 4-bit của unsloth "
            "không khớp bộ nạp của vLLM, phải đổi phiên bản chứ không phải tải lại.") from _e
    else:
        raise

SAMPLING = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                          repetition_penalty=REPETITION_PENALTY, seed=RANDOM_SEED)
LORA_REQUEST = None          # nấc 3 trở đi có thể gán adapter đã SFT vào đây

from collections import Counter as _Counter
LY_DO_DUNG = _Counter()          # finish_reason: "stop" = tự kết thúc, "length" = BỊ CẮT
LY_DO_THEO_BUOC = {}             # desc → Counter riêng, để tách bước 1 với bước 2


def ty_le_bi_cat():
    """Phần trăm lượt sinh bị cắt vì chạm max_tokens, TÍNH TỪ LẦN reset gần nhất."""
    t = sum(LY_DO_DUNG.values())
    return (LY_DO_DUNG.get("length", 0) / t) if t else 0.0


def bi_cat_theo_buoc():
    """Tỉ lệ bị cắt TÁCH RIÊNG cho bước 1 và bước 2.

    Phải tách vì prompt bước 2 (self-eval ở nấc 4, ACE ở nấc 5) chứa NGUYÊN lời giải
    bước 1, nên dài hơn bước 1 rất nhiều. Bước 2 bị cắt nhiều hơn nghĩa là phương pháp
    bị PHA LOÃNG — mất cơ hội sửa, chứ không phải sửa sai. Con số gộp chung không phân
    biệt được hai chuyện đó.

    Gom theo đuôi của desc ("vòng3/step1" và "vòng7/step1" cùng vào "step1").
    """
    gom = {}
    for k, c in LY_DO_THEO_BUOC.items():
        gom.setdefault(k.rsplit("/", 1)[-1], _Counter()).update(c)
    return {b: {"n": sum(c.values()), "bi_cat": c.get("length", 0),
                "ty_le": round(c.get("length", 0) / max(1, sum(c.values())), 4)}
            for b, c in sorted(gom.items())}


def in_bi_cat_theo_buoc():
    d = bi_cat_theo_buoc()
    if not d:
        return
    # In cả khi chỉ có MỘT bước: nấc 1 và 2 cũng cần biết tỉ lệ chạm trần của mình,
    # nếu không thì mãi tới nấc 4 mới thấy con số đó.
    print("   Bị cắt vì trần token, tách theo bước:")
    for b, v in d.items():
        print(f"     {b:<10}{v['ty_le']:>7.1%}  ({v['bi_cat']}/{v['n']} lượt)")
    if "step2" in d and "step1" in d and d["step2"]["ty_le"] > d["step1"]["ty_le"] + 0.02:
        print("     ⚠ bước 2 bị cắt nhiều hơn bước 1 → hiệu quả của phương pháp đang bị")
        print("       PHA LOÃNG (mất cơ hội sửa). Hiệu số đo được là cận DƯỚI.")


def dat_lai_bo_dem():
    """Gọi ngay trước mỗi nấc. Không gọi thì tỉ lệ là cộng dồn cả phiên — gồm cả
    lượt warmup và (ở nấc 5) toàn bộ pha A, không phản ánh nấc đang đo."""
    LY_DO_DUNG.clear()
    LY_DO_THEO_BUOC.clear()


def generate(prompts, sampling_params=None, desc=None, batch_size=None):
    """Sinh theo lô qua vLLM — như vòng lặp trong notebook cũ."""
    if not prompts:
        return []
    sp = sampling_params or SAMPLING
    bs = batch_size or BATCH_SIZE
    outs, t0 = [], time.time()
    nb = (len(prompts) + bs - 1) // bs
    for i in range(nb):
        chunk = prompts[i*bs:(i+1)*bs]
        kw = {"sampling_params": sp}
        if LORA_REQUEST is not None:
            kw["lora_request"] = LORA_REQUEST
        _res = model.fast_generate(chunk, **kw)
        for o in _res:                      # đếm lý do dừng để biết có bị cắt không
            for _x in o.outputs:            # sinh nhiều mẫu thì đếm CẢ K mẫu
                _r = getattr(_x, "finish_reason", "?")
                LY_DO_DUNG[_r] += 1
                if desc:
                    LY_DO_THEO_BUOC.setdefault(desc, _Counter())[_r] += 1
        # 1 mẫu → trả chuỗi (y như cũ); nhiều mẫu → trả list[str] cho self-consistency.
        outs.extend((o.outputs[0].text if len(o.outputs) == 1
                     else [_x.text for _x in o.outputs]) for o in _res)
        if desc:
            el = time.time() - t0
            print(f"    {desc}: lô {i+1}/{nb} | {el:.0f}s | "
                  f"ETA {el/(i+1)*(nb-i-1):.0f}s", end="\r")
    gc.collect(); torch.cuda.empty_cache()
    if desc:
        _b = LY_DO_THEO_BUOC.get(desc, _Counter())
        _c, _n = _b.get("length", 0), max(1, sum(_b.values()))
        print(f"    {desc}: xong {len(prompts)} prompt trong {time.time()-t0:.0f}s"
              f" | bị cắt vì trần token: {_c/_n:.1%} ({_c} lượt)" + " "*8)
    return outs

_t = torch.cuda.get_device_properties(0).total_memory/1024**3
print(f"[MODEL] ✅ sẵn sàng sau {(time.time()-_t_nap)/60:.1f} phút | "
      f"VRAM {_t - torch.cuda.mem_get_info()[0]/1024**3:.1f}/{_t:.1f} GB")
_ = generate(["xin chào"], SamplingParams(temperature=0, max_tokens=4))
print("[WARMUP] ✅")

In [ ]:
prompt_kit = PromptKit(tokenizer=tokenizer, model_name=MODEL_NAME)
PROMPT_LEVEL = "engineered"
USE_SELFEVAL = False

_think = getattr(prompt_kit, "enable_thinking", None)
print(f"[PROMPT] mức = {PROMPT_LEVEL} | self-eval = {USE_SELFEVAL} | "
      f"suy nghĩ = {'template tự quyết (Qwen3: BẬT)' if _think is None else _think}")
if _think is False:
    print("[PROMPT] ⚠ suy nghĩ đang TẮT — lệch bản tham chiếu, PA sẽ hụt "
          "~10 điểm. Dấu hiệu: 497 mẫu chạy xong trong ~1 phút.")
print(f"         thang lồng nhau: basic={len(prompt_kit.BASIC_SYSTEM_PROMPT)} ký tự"
      f" ⊂ no_fewshot={len(prompt_kit.NO_FEWSHOT_SYSTEM_PROMPT)}"
      f" ⊂ engineered={len(prompt_kit.ENGINEERED_SYSTEM_PROMPT)}"
      f" | self-eval={len(prompt_kit.SELF_EVAL_SYSTEM_PROMPT)}")

# Đo bằng tokenizer THẬT trên 40 mẫu có ngữ cảnh DÀI NHẤT
BUDGET = MAX_SEQ_LENGTH - MAX_TOKENS
_clen = lambda s: (len(" ".join(s.get("pre_text") or [])) +
                   len(" ".join(s.get("post_text") or [])) + len(str(s.get("table") or "")))
_probe = sorted(test_all, key=_clen, reverse=True)[:40]
_bul = "\n".join(["- Khi hỏi tốc độ tăng trưởng, dùng subtract(gia_tri_moi, gia_tri_cu), "
                  "divide(#0, gia_tri_cu)."] * 7)
# Lời giải bước 1 dài nhất có thể là đúng MAX_TOKENS token (model sinh chạm trần).
# Phải đo ở mức đó, không thì bật suy nghĩ vào là prompt bước 2 tràn ngân sách.
_unit = "Phân tích chi tiết từng bước của bảng số liệu. "
_prev = _unit * max(1, MAX_TOKENS // max(1, len(tokenizer(_unit).input_ids)))
_prev += "\n```plaintext\nprogram: divide(1,2)\nanswer: 0.5\n```"

def _measure():
    a = [len(tokenizer(prompt_kit.step1(s, _bul, level=PROMPT_LEVEL)).input_ids)
         for s in _probe]
    b = ([len(tokenizer(prompt_kit.step2(s, _prev, _bul)).input_ids) for s in _probe]
         if USE_SELFEVAL else [0])
    return a, b

_a, _b = _measure()
print(f"[PROMPT] (40 mẫu dài nhất) step1 max={max(_a)} | step2 max={max(_b)} | "
      f"ngân sách={BUDGET}")

if max(max(_a), max(_b)) > BUDGET:
    prompt_kit.max_prev_chars = 3000
    _cap = _clen(_probe[0])
    for _ in range(6):
        _cap = int(_cap * 0.80)
        prompt_kit.max_ctx_chars = max(1200, _cap)
        _a, _b = _measure()
        if max(max(_a), max(_b)) <= BUDGET:
            break
    assert max(max(_a), max(_b)) <= BUDGET, "Không cắt đủ — giảm MAX_TOKENS hoặc dùng GPU lớn hơn."
    _hit = sum(1 for s in test_all if _clen(s) > prompt_kit.max_ctx_chars)
    print(f"[PROMPT] ⚠ đã bật cắt ngữ cảnh (max_ctx_chars={prompt_kit.max_ctx_chars}); "
          f"{_hit}/{len(test_all)} mẫu bị cắt ({_hit/len(test_all)*100:.1f}%)")
    print(f"[PROMPT]   GHI LẠI con số này khi báo cáo.")
else:
    print("[PROMPT] ✅ mọi prompt đều lọt ngân sách, không cần cắt")

## §3. Chọn tập train để sinh dữ liệu

`SFT_TRAIN_SUBSET` là số mẫu train đem ra chạy sinh. Vì chỉ giữ lại mẫu model làm đúng
(~50–60 % theo kết quả nấc 2), lấy 2000 mẫu sẽ cho khoảng 1000–1200 mẫu huấn luyện —
cùng cỡ với 2237 mẫu mà lần chạy trước dùng cho Phi-4.

Mẫu được lấy **phân tầng theo số phép toán** để dữ liệu huấn luyện không chỉ toàn bài 1 bước.

In [ ]:
SFT_TRAIN_SUBSET = 2000        # đặt None để dùng cả 2993 mẫu (lâu hơn ~50%)
DROP_NOISY_GOLD  = True        # bỏ mẫu multiply(#n,100) và gold lỗi cú pháp
ACCEPT           = "pa_or_ea"  # 'pa' chặt nhất | 'ea' | 'pa_or_ea'
ADD_GOLD_FALLBACK = False      # bật = quay lại rủi ro overfit của cách cũ

sft_train = data.stratified_sample(train_all, SFT_TRAIN_SUBSET, seed=RANDOM_SEED)
print(f"[SFT] lấy {len(sft_train)} mẫu train")
print("  phân tầng:", dict(sorted(Counter(min(dsl.n_ops(s['qa']['program']), 5)
                                          for s in sft_train).items())))
print("  nguồn    :", {k: len(v) for k, v in data.split_by_source(sft_train).items()})
_nz = sum(1 for s in sft_train if data.is_noisy_gold(s))
print(f"  nhãn nhiễu: {_nz} ({_nz/len(sft_train)*100:.1f}%)"
      f"{' → sẽ bị lọc' if DROP_NOISY_GOLD else ' → GIỮ LẠI (không khuyến nghị)'}")

## §4. Sinh lời giải trên tập train

Chạy đúng pipeline của nấc 2 (prompt có cấu trúc, một bước) trên tập train.
Đây là bước tốn thời gian nhất của phần A.

In [ ]:
dat_lai_bo_dem()          # lượt sinh này dựng DỮ LIỆU SFT — bị cắt ở đây là
                          # mất đúng những mẫu khó, phải đo riêng

# ── ĐIỂM LƯU ───────────────────────────────────────────────────────────────────
# Lượt sinh này tốn ~45 phút và trước đây KHÔNG hề được lưu, nên bất kỳ lỗi nào ở
# dòng phân tích ngay bên dưới là mất trắng cả 45 phút. Đã xảy ra thật: tập train
# có 5 nhãn vàng cụt dấu ngoặc, `summarize()` ném ValueError sau khi sinh xong.
# Giờ: sinh xong GHI NGAY, phân tích sau. Chạy lại cell này thì dùng lại bản đã ghi.
_CKPT = os.path.join(OUTPUT_DIR, "sft_data", f"train_rows_{len(sft_train)}.jsonl")
os.makedirs(os.path.dirname(_CKPT), exist_ok=True)
_VAN = {"n": len(sft_train), "subset": SFT_TRAIN_SUBSET, "level": "engineered",
        "seed": RANDOM_SEED, "max_tokens": MAX_TOKENS, "model": MODEL_NAME}

train_rows, _da_sinh = None, False
if os.path.exists(_CKPT):
    _d = [json.loads(l) for l in open(_CKPT, encoding="utf-8") if l.strip()]
    if _d and _d[0].get("__van__") == _VAN:
        train_rows = _d[1:]
        print(f"[ĐIỂM LƯU] dùng lại {len(train_rows)} lời giải đã sinh — "
              f"bỏ qua ~45 phút GPU")
        print(f"           {_CKPT}")
    else:
        print("[ĐIỂM LƯU] ⚠ cấu hình đã đổi so với bản đã ghi — sinh lại từ đầu")

_t0 = time.time()
if train_rows is None:
    train_rows = pipeline.run_pipeline(
        sft_train, prompt_kit, generate,
        prompt_level="engineered", use_selfeval=False,
        sp_step1=SAMPLING, desc="sinh-train", keep_raw=True)
    _da_sinh = True
    # GHI TRƯỚC, PHÂN TÍCH SAU. Thứ tự hai khối này đáng đúng 45 phút GPU.
    _tmp = _CKPT + ".tmp"
    with open(_tmp, "w", encoding="utf-8") as f:
        f.write(json.dumps({"__van__": _VAN}, ensure_ascii=False) + "\n")
        for _r in train_rows:
            f.write(json.dumps(_r, ensure_ascii=False, default=str) + "\n")
    os.replace(_tmp, _CKPT)
    print(f"\n[ĐIỂM LƯU] đã ghi {len(train_rows)} lời giải → {_CKPT}")

_m_train = pipeline.summarize(train_rows, "train (để dựng SFT)")
if _da_sinh:
    print(f"\n  Thời gian: {(time.time()-_t0)/60:.1f} phút")
pipeline.print_summary(_m_train)
if _da_sinh:
    in_bi_cat_theo_buoc()
print(f"\n  → Tỉ lệ làm đúng trên train ≈ tỉ lệ mẫu sẽ vào được dữ liệu SFT.")


## §5. Lọc và ghi dữ liệu SFT

Cell dưới in rõ **mỗi mẫu bị loại vì lý do gì** — để kiểm chứng rằng việc lọc là có cơ sở
chứ không phải cắt bừa dữ liệu khó.

In [ ]:
records, stats_build = sft.build_sft_records(
    train_rows, sft_train, prompt_kit,
    level="engineered", accept=ACCEPT,
    add_gold_fallback=ADD_GOLD_FALLBACK, drop_noisy_gold=DROP_NOISY_GOLD)

print(f"{'═'*62}\n  DỰNG DỮ LIỆU SFT\n{'═'*62}")
for k, v in sorted(stats_build.items(), key=lambda x: -x[1]):
    print(f"  {k:<28}{v:>6}")
print(f"\n  → giữ lại {len(records)} mẫu huấn luyện")

st = sft.sft_data_stats(records)
print(f"\n  Phân bố theo số phép toán: {st['theo_so_phep']}")
print(f"  Nguồn đích               : {st['theo_nguon']}")
print(f"  Độ dài (ký tự) p50/p95/max: {st['ky_tu_p50']} / {st['ky_tu_p95']} / {st['ky_tu_max']}")

assert len(records) >= 200, (
    f"Chỉ {len(records)} mẫu — quá ít để fine-tune. Tăng SFT_TRAIN_SUBSET, "
    f"hoặc đổi ACCEPT='ea', hoặc kiểm tra lại nấc 2.")

SFT_JSONL = os.path.join(OUTPUT_DIR, "sft_data", f"qwen3_sft_{STAMP}.jsonl")
sft.write_jsonl(records, SFT_JSONL)
with open(os.path.join(OUTPUT_DIR, "sft_data", "latest.txt"), "w") as f:
    f.write(SFT_JSONL)
print(f"\n[SAVE] {SFT_JSONL}")

# Hồ sơ dựng dữ liệu: bao nhiêu mẫu bị loại vì lý do gì, cấu hình nào sinh ra nó.
_build = os.path.join(OUTPUT_DIR, "sft_data", f"qwen3_sft_{STAMP}_build.json")
json.dump({"stamp": STAMP, "jsonl": SFT_JSONL, "n_records": len(records),
           "loc": stats_build, "phan_bo": st,
           "cau_hinh": {"SFT_TRAIN_SUBSET": SFT_TRAIN_SUBSET, "ACCEPT": ACCEPT,
                        "DROP_NOISY_GOLD": DROP_NOISY_GOLD,
                        "ADD_GOLD_FALLBACK": ADD_GOLD_FALLBACK,
                        "n_train_da_sinh": len(train_rows)},
           "metrics_tren_train": pipeline.summarize(train_rows, "train (sinh de loc)"),
           "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
           "env": run_env()},
          open(_build, "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
print(f"[SAVE] {_build}   ← hồ sơ lọc dữ liệu")

print(f"\n{'═'*62}\n  MỘT MẪU HUẤN LUYỆN\n{'═'*62}")
_ex = records[0]["messages"]
print(f"  [system] {len(_ex[0]['content'])} ký tự (prompt có cấu trúc, cắt bớt khi in)")
print(f"  [user]   {_ex[1]['content'][:300]}...")
print(f"  [assistant] ↓ đây là thứ model sẽ học sinh ra")
print(_ex[2]['content'][:900])

---

# ⚠ RESTART RUNTIME TẠI ĐÂY

**Runtime → Restart session**, rồi chạy tiếp từ §6.

Lý do: engine vLLM giữ VRAM rất chặt và không nhả đủ cho việc huấn luyện trong cùng phiên.
Dữ liệu SFT đã ghi ra Drive nên phần B đọc lại được, không mất gì.

---

# PHẦN B — Huấn luyện LoRA

## §6. Môi trường (chạy lại sau restart)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, sft, stats
from vinumqa.prompts import PromptKit

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

# Thang prompt LỒNG NHAU: basic ⊂ no_fewshot ⊂ engineered — mỗi nấc thêm đúng một khối
# cắt ra từ prompt hoàn chỉnh, phần chung giống nhau từng ký tự.
LADDER = [
    ("01_basic",           "Nấc 1 — prompt cơ bản (danh sách phép toán + yêu cầu)"),
    ("02_prompt_eng",      "Nấc 2 — prompt hoàn chỉnh (+ hướng dẫn từ khoá + few-shot)"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("04_selfeval_sft",    "Nấc 4b — + self-eval (model SFT)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05_ace_sft",         "Nấc 5b — + ACE (model SFT)"),
    ("05c_ace_basic_base", "Nấc 5c — ACE trên prompt cơ bản"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("06_comb_F_A",        "Tổ hợp — SFT + ACE (không self-eval)"),
    ("08_tu_nhat_quan",    "Mới — self-consistency K mẫu (ví dụ cố định)"),
    ("09_vidu_dong",       "Mới — self-consistency + ví dụ truy hồi"),
    ("06_comb_E_A_moi",    "Mục tiêu — prompt + ACE + phương pháp mới"),
    ("06_comb_F_A_moi",    "Mục tiêu — SFT + ACE + phương pháp mới"),
]
LADDER_LABEL = dict(LADDER)

# Những biến phải đi kèm kết quả thì mới truy lại được về sau.
_CFG_KEYS = ("MODEL_NAME", "MODEL_TAG", "TEMPERATURE", "MAX_TOKENS", "REPETITION_PENALTY",
             "MAX_SEQ_LENGTH", "BATCH_SIZE", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "RANDOM_SEED")


def run_env():
    """Môi trường THẬT lúc chạy: commit, GPU, phiên bản thư viện.

    Chỉ đọc thư viện đã nạp (``sys.modules``) chứ không import thêm — vừa nhanh,
    vừa báo đúng những gì thật sự được dùng.
    """
    env = {"python": sys.version.split()[0]}
    try:                                   # bản code nào sinh ra kết quả này
        def _g(*a):
            return subprocess.run(["git", "-C", REPO_DIR, *a],
                                  capture_output=True, text=True).stdout.strip()
        env["commit"] = _g("rev-parse", "--short", "HEAD")
        env["branch"] = _g("rev-parse", "--abbrev-ref", "HEAD")
        env["dirty"] = bool(_g("status", "--porcelain"))
    except Exception:
        pass
    _torch = sys.modules.get("torch")
    if _torch is not None:
        env["torch"] = getattr(_torch, "__version__", "?")
        try:
            if _torch.cuda.is_available():
                _p = _torch.cuda.get_device_properties(0)
                env["gpu"] = _p.name
                env["vram_gb"] = round(_p.total_memory / 1024**3, 1)
                env["cc"] = f"{_p.major}.{_p.minor}"
            else:
                env["gpu"] = "CPU"
        except Exception:
            pass
    else:
        env["gpu"] = "CPU (không nạp torch)"
    for _lib in ("transformers", "trl", "peft", "vllm", "unsloth",
                 "sentence_transformers", "numpy"):
        _m = sys.modules.get(_lib)
        if _m is not None and hasattr(_m, "__version__"):
            env[_lib] = _m.__version__
    return env


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "meta":  f"{stage}_meta.json"}[kind])


# Cấu hình CHUẨN của cả thang bậc — đo trên A100 40GB.
# MAX_SEQ_LENGTH đổi theo GPU (A100 15000 / L4 13500 / T4 8192), mà đổi GPU là đổi
# thành phần lô, đổi kernel, đổi luôn token được lấy mẫu ở temperature 0.1. Hai lần
# chạy khác max_seq KHÔNG so thẳng được, nên phải ghi sang tên nấc khác.
MAX_TOKENS_CHUAN, MAX_SEQ_CHUAN = 4096, 17000



def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: jsonl + meta, kèm một file output thô.

    Chạy với ``MAX_TOKENS`` khác mức chuẩn thì tự ghi sang tên nấc khác. Đổi trần sinh
    là đổi cấu hình, kết quả không so thẳng với thang bậc được — mà nếu cứ ghi đè lên
    tên cũ thì mất luôn bản chuẩn, không lấy lại được nếu không chạy lại GPU.
    """
    _hau_to = ""
    _mt, _ms = globals().get("MAX_TOKENS"), globals().get("MAX_SEQ_LENGTH")
    if _mt and _mt != MAX_TOKENS_CHUAN:
        _hau_to += f"_tok{_mt}"
    if _ms and _ms != MAX_SEQ_CHUAN:
        _hau_to += f"_seq{_ms}"
    if _hau_to and not stage.endswith(_hau_to):
        stage = f"{stage}{_hau_to}"
        if not quiet:
            print(f"[GHI] ⚠ cấu hình khác chuẩn (max_tokens={_mt}, max_seq={_ms}, "
                  f"GPU={globals().get('_GPU', '?')}) → ghi sang nấc '{stage}'.")
            print( "       Kết quả khác GPU/khác trần KHÔNG so thẳng với thang bậc chuẩn.")
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            "enable_thinking": getattr(globals().get("prompt_kit", None),
                                       "enable_thinking", "?"),
            "ty_le_bi_cat_token": (round(ty_le_bi_cat(), 4)
                                   if "ty_le_bi_cat" in globals() else None),
            "bi_cat_theo_buoc": (bi_cat_theo_buoc()
                                if "bi_cat_theo_buoc" in globals() else None),
            "nap_an_toan": bool(globals().get("NAP_AN_TOAN", False)),
            "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
            "env": run_env(),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'meta')}")
    return stage                      # tên THẬT, có thể khác tên truyền vào


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else "máy cá nhân"
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

In [ ]:
# Tìm file dữ liệu SFT vừa dựng ở phần A
_latest = os.path.join(OUTPUT_DIR, "sft_data", "latest.txt")
if os.path.exists(_latest):
    SFT_JSONL = open(_latest).read().strip()
else:
    _cands = sorted(__import__("glob").glob(
        os.path.join(OUTPUT_DIR, "sft_data", "qwen3_sft_*.jsonl")))
    assert _cands, "Không thấy dữ liệu SFT — chạy phần A trước."
    SFT_JSONL = _cands[-1]

_n = sum(1 for _ in open(SFT_JSONL, encoding="utf-8"))
print(f"[SFT] dữ liệu: {SFT_JSONL}")
print(f"[SFT] {_n} mẫu")

## §7. Nạp model ở chế độ huấn luyện

Khác phần A ở hai chỗ: **không** bật `fast_inference` (đó là engine suy luận), và gắn thêm
LoRA adapter qua `get_peft_model`.

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3-8B"
MODEL_TAG  = "Qwen3-8B"
_VRAM = torch.cuda.get_device_properties(0).total_memory / 1024**3
_CC = torch.cuda.get_device_capability(0)
BF16 = _CC[0] >= 8                       # T4 (SM75) không có bfloat16

# Đích huấn luyện là lời giải bước 1. Trần sinh đã nâng lên 4096 nên bản ghi dài nhất
# ~9.100 token (system 1.7k + ngữ cảnh 3.3k + lời giải 4.1k) — để 8192 là LỌC BỎ đúng
# những mẫu khó nhất, khiến SFT chỉ học từ mẫu dễ. `training_config` chia batch theo
# NGÂN SÁCH TOKEN dựa trên chính con số này.
MAX_SEQ_LENGTH = 8192

# ═══ CỔNG VRAM ═══════════════════════════════════════════════════════════════════
# PHẦN A đã dựng engine vLLM. Không restart runtime thì VRAM còn bị giữ, và accelerate
# ÂM THẦM đẩy embed_tokens sang CPU thay vì báo thiếu bộ nhớ. Mãi tới step huấn luyện
# ĐẦU TIÊN nó mới nổ "index is on cuda:0, different from other tensors on cpu", kèm
# traceback 29 tầng không chỉ ra nguyên nhân thật. Chặn ở đây tốn 1 giây.
def _do_vram():
    _f, _t = torch.cuda.mem_get_info(0)
    return _f / 1024**3, _t / 1024**3


_free, _tong = _do_vram()
print(f"[VRAM] trống {_free:.1f} / {_tong:.1f} GB")

# Thử dọn TRƯỚC khi bắt restart: nếu ô này chạy ngay sau PHẦN A trong cùng phiên thì
# `model`/`tokenizer` của vLLM vẫn còn trong globals. Tốn 5 giây, không mất gì nếu
# không ăn thua.
if _free < max(12.0, 0.6 * _tong):
    print("[VRAM] thử giải phóng engine của PHẦN A ...")
    try:
        from vllm.distributed.parallel_state import (
            destroy_distributed_environment, destroy_model_parallel)
        destroy_model_parallel()
        destroy_distributed_environment()
    except Exception:                                # noqa: BLE001
        pass
    for _ten in ("model", "tokenizer", "trainer", "llm", "generate"):
        globals().pop(_ten, None)
    gc.collect()
    torch.cuda.empty_cache()
    _free, _tong = _do_vram()
    print(f"[VRAM] sau khi dọn: trống {_free:.1f} / {_tong:.1f} GB")

# Ngưỡng theo TỈ LỆ chứ không chỉ theo con số tuyệt đối: GPU sạch luôn trống >95 %,
# còn engine vLLM nằm lại thì chỉ còn ~14 %. Tỉ lệ đúng cho mọi cỡ card.
assert _free >= max(12.0, 0.6 * _tong), (
    f"Chỉ còn {_free:.1f}/{_tong:.1f} GB trống — engine vLLM của PHẦN A vẫn nằm đó.\n"
    f"    1. Runtime ▸ Restart session\n"
    f"    2. Chạy lại TỪ Ô #10 — ô bootstrap ngay dưới ô chữ 'RESTART RUNTIME TẠI"
    f" ĐÂY'. KHÔNG phải từ ô #1, và KHÔNG Ctrl+F9.\n"
    f"    PHẦN A đã xong, dữ liệu SFT nằm trên Drive — ô #11 tự đọc lại.")


def _nap(che_do_ckpt):
    m, tk = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = None,
        load_in_4bit   = True,
    )
    return FastLanguageModel.get_peft_model(
        m, **dict(sft.lora_config(), use_gradient_checkpointing=che_do_ckpt)), tk


# ═══ CỔNG THIẾT BỊ ═══════════════════════════════════════════════════════════════
# Một forward tí hon, 2 giây, TRƯỚC khi dựng dataset và trainer. Gradient checkpointing
# "unsloth" tiết kiệm VRAM bằng cách đẩy embedding sang CPU — tuỳ bản unsloth/torch mà
# nó lệch thiết bị. Thử trước; hỏng thì tự lùi về bản chuẩn của HF, không phải chạy lại.
GC_MODE = None
for _gc in ("unsloth", True):
    model, tokenizer = _nap(_gc)
    _cpu = [n for n, t in list(model.named_parameters()) + list(model.named_buffers())
            if t.device.type != "cuda"]
    assert not _cpu, (
        f"{len(_cpu)} tensor nằm trên CPU (vd {_cpu[:3]}) — accelerate đã offload vì "
        f"thiếu VRAM. Restart session rồi chạy lại từ ô đầu của PHẦN B.")
    try:
        with torch.no_grad():
            model(tokenizer("kiểm tra thiết bị", return_tensors="pt")
                  .input_ids.to("cuda"))
    except RuntimeError as _e:
        if "same device" not in str(_e) or _gc is True:
            raise
        print(f"[GPU] ⚠ checkpointing 'unsloth' lệch thiết bị: {_e}")
        print("[GPU] → nạp lại với gradient checkpointing chuẩn của HF")
        del model, tokenizer
        gc.collect(); torch.cuda.empty_cache()
        continue
    GC_MODE = _gc
    break

print(f"[GPU] {torch.cuda.get_device_name(0)} | {_VRAM:.1f} GB | bf16={BF16}")
print(f"[GPU] ✅ forward thử chạy được | gradient checkpointing = {GC_MODE!r}")
print(f"[LoRA] {dict(sft.lora_config(), use_gradient_checkpointing=GC_MODE)}")


## §8. Chuẩn bị dataset

Chia 90/10 train/val, áp chat template của Qwen3, và **lọc bỏ mẫu vượt `max_seq_length`**
(giống bước lọc `full_tokens <= MAX_TOKEN` của notebook cũ).

In [ ]:
from datasets import load_dataset

full_ds = load_dataset("json", data_files=SFT_JSONL, split="train")
split = full_ds.train_test_split(test_size=0.1, seed=3407)
train_ds, val_ds = split["train"], split["test"]

def _format(ex):
    # Dùng chat template có sẵn của Qwen3 — KHÔNG ghi đè như notebook cũ làm với phi-3
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False,
                                                  add_generation_prompt=False)}

train_ds = train_ds.map(_format)
val_ds = val_ds.map(_format)

def _ntok(ex):
    return {"n_tokens": len(tokenizer(ex["text"]).input_ids)}

train_ds = train_ds.map(_ntok)
val_ds = val_ds.map(_ntok)

_before = (len(train_ds), len(val_ds))
train_ds = train_ds.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH)
val_ds = val_ds.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH)
print(f"[DATA] train {_before[0]} → {len(train_ds)} | val {_before[1]} → {len(val_ds)}"
      f"  (lọc mẫu > {MAX_SEQ_LENGTH} token)")
_mat = 1 - len(train_ds) / max(1, _before[0])
print(f"[DATA] mất {_mat:.1%} vì quá dài" + ("" if _mat <= 0.05 else
      "  ⚠ trên 5 % — SFT đang bỏ qua nhóm mẫu dài/khó, nâng MAX_SEQ_LENGTH ở §7"))

import numpy as _np
_t = _np.array(train_ds["n_tokens"])
print(f"[DATA] token: p50={int(_np.percentile(_t,50))} p95={int(_np.percentile(_t,95))} "
      f"max={_t.max()}")
assert len(train_ds) >= 100, "Quá ít mẫu sau khi lọc."

## §9. Huấn luyện

In [ ]:
import inspect, dataclasses
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback, TrainerCallback

ADAPTER_DIR = os.path.join(OUTPUT_DIR, "sft_adapter_qwen3")
CKPT_DIR = os.path.join(OUTPUT_DIR, "sft_checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

cfg = sft.training_config(_VRAM, len(train_ds), epochs=3, output_dir=CKPT_DIR,
                          bf16=BF16, max_seq=MAX_SEQ_LENGTH)
meta = cfg.pop("_meta")
print(f"[TRAIN] batch/thiết bị={cfg['per_device_train_batch_size']} × "
      f"accum={cfg['gradient_accumulation_steps']} → batch hiệu dụng={meta['effective_batch']}")
print(f"[TRAIN] {meta['steps_per_epoch']} step/epoch × 3 epoch = {meta['total_steps']} step "
      f"| eval mỗi {meta['eval_every']} step")

print(f"[TRAIN] ngân sách {meta['token_moi_lo']} token/micro-batch "
      f"→ riêng tensor logits ~{meta['logits_gb']} GB (bf16)")
assert meta["vua_vram"], (
    f"GPU {_VRAM:.0f} GB không đủ cho chuỗi {MAX_SEQ_LENGTH} token: ngay ở batch=1, "
    f"riêng logits đã {meta['logits_gb']} GB bf16 và cross-entropy còn upcast fp32 "
    f"(×2) nữa. Dùng GPU lớn hơn, hoặc hạ MAX_SEQ_LENGTH ở §7 và chấp nhận lọc mất "
    f"nhóm mẫu dài. Đừng chạy tiếp — sẽ tràn giữa chừng.")

# ═══ CỔNG BỘ NHỚ ═════════════════════════════════════════════════════════════════
# Đo ĐỈNH THẬT bằng một forward+backward trên lô ĐẦY, ~30 giây. Tràn VRAM ở giữa lượt
# huấn luyện là mất cả giờ; tràn ở đây thì hạ batch rồi đi tiếp, không phải sửa code.
# Lô đầy là trường hợp XẤU NHẤT có thật: packing=False nên mỗi lô đệm theo chuỗi dài
# nhất trong lô, và một chuỗi dài có thể rơi vào bất kỳ lô nào sau khi trộn.
# (Chưa tính optimizer state, nhưng paged_adamw_8bit chỉ giữ ~87 MB cho 43,6M tham số
#  LoRA nên phần đó không đổi kết luận.)
def _thu_bo_nho(bs):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    _ids = torch.randint(0, 1000, (bs, MAX_SEQ_LENGTH), device="cuda")
    _out = model(input_ids=_ids, labels=_ids)
    _out.loss.backward()
    _dinh = torch.cuda.max_memory_allocated() / 1024 ** 3
    model.zero_grad(set_to_none=True)
    del _out, _ids
    gc.collect(); torch.cuda.empty_cache()
    return _dinh

while True:
    _bs = cfg["per_device_train_batch_size"]
    try:
        _dinh = _thu_bo_nho(_bs)
    except RuntimeError as _e:
        if "out of memory" not in str(_e).lower():
            raise
        model.zero_grad(set_to_none=True)
        gc.collect(); torch.cuda.empty_cache()
        assert _bs > 1, ("Tràn VRAM ngay ở batch=1 — hạ MAX_SEQ_LENGTH hoặc dùng GPU "
                         "lớn hơn. Đừng chạy tiếp, sẽ tràn giữa chừng.")
        # accum tính LẠI từ đích 16, không phải nhân đôi. Nhân đôi chỉ đúng khi bs
        # là luỹ thừa của 2; với bs=5 thì 5×3=15 thành 2×6=12, batch hiệu dụng lệch
        # → `_sig` đổi → checkpoint của chính lần chạy này bị coi là của lần khác.
        _bs_moi = max(1, _bs // 2)
        cfg["per_device_train_batch_size"] = _bs_moi
        cfg["per_device_eval_batch_size"] = _bs_moi
        cfg["gradient_accumulation_steps"] = max(1, round(16 / _bs_moi))
        print(f"[VRAM] ⚠ tràn ở batch={_bs} → hạ còn {_bs_moi}, accum "
              f"{cfg['gradient_accumulation_steps']} — batch hiệu dụng giữ ở "
              f"{_bs_moi * cfg['gradient_accumulation_steps']}")
        continue
    print(f"[VRAM] đỉnh ở lô ĐẦY ({_bs}×{MAX_SEQ_LENGTH} token) = "
          f"{_dinh:.1f} / {_VRAM:.1f} GB")
    break

# Cổng bộ nhớ có thể đã đổi batch → mọi con số dẫn xuất phải tính lại, vì `_sig`
# bên dưới dùng chúng để quyết định có nối tiếp checkpoint hay không.
meta["effective_batch"] = (cfg["per_device_train_batch_size"]
                           * cfg["gradient_accumulation_steps"])
meta["steps_per_epoch"] = max(1, len(train_ds) // meta["effective_batch"])
meta["total_steps"] = meta["steps_per_epoch"] * 3

class ClearCache(TrainerCallback):
    def on_evaluate(self, args, state, control, **kw):
        gc.collect(); torch.cuda.empty_cache()

_CB = [EarlyStoppingCallback(early_stopping_patience=3), ClearCache()]

# trl đổi API giữa các bản: ≤0.22 nhận tokenizer / dataset_text_field / max_seq_length
# thẳng ở SFTTrainer; từ 0.23 dồn hết vào SFTConfig và đổi tên thành processing_class /
# max_length. Dò chữ ký thật để chạy được với cả hai, khỏi phụ thuộc bản nào được cài.
if "dataset_text_field" in inspect.signature(SFTTrainer.__init__).parameters:
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=train_ds, eval_dataset=val_ds,
        args=TrainingArguments(**cfg),
        dataset_text_field="text", packing=False, max_seq_length=MAX_SEQ_LENGTH,
        callbacks=_CB)
    print("[TRL] API cũ — SFTTrainer nhận dataset_text_field")
else:
    from trl import SFTConfig
    _f = {x.name for x in dataclasses.fields(SFTConfig)}
    _len_key = "max_length" if "max_length" in _f else "max_seq_length"
    _extra = {"dataset_text_field": "text", "packing": False, _len_key: MAX_SEQ_LENGTH}
    trainer = SFTTrainer(
        model=model, processing_class=tokenizer,
        train_dataset=train_ds, eval_dataset=val_ds,
        args=SFTConfig(**cfg, **_extra),
        callbacks=_CB)
    print(f"[TRL] API mới — SFTConfig, độ dài đặt qua '{_len_key}'")

# Tiếp tục từ checkpoint nếu Colab đã ngắt giữa chừng — nhưng CHỈ khi checkpoint đó
# thuộc đúng lần huấn luyện này. Nối tiếp checkpoint của một bộ dữ liệu khác là hỏng
# âm thầm: sai dữ liệu, sai số step, adapter ra không phải cái mình tưởng.
_sig = {"sft_data": os.path.basename(SFT_JSONL), "n_train": len(train_ds),
        "effective_batch": meta["effective_batch"], "total_steps": meta["total_steps"]}
_sig_path = os.path.join(CKPT_DIR, "run_signature.json")
try:
    _old_sig = json.load(open(_sig_path, encoding="utf-8"))
except Exception:
    _old_sig = None

_ck = sorted([d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")],
             key=lambda x: int(x.split("-")[-1]))
if _ck and _old_sig != _sig:
    try:
        _moved = f"{CKPT_DIR}_cu_{datetime.now().strftime('%Y%m%d_%H%M')}"
        os.rename(CKPT_DIR, _moved)
        os.makedirs(CKPT_DIR, exist_ok=True)
        print(f"[TRAIN] ⚠ checkpoint cũ thuộc lần chạy khác → đã dời sang {_moved}")
    except OSError:
        print("[TRAIN] ⚠ checkpoint cũ thuộc lần chạy khác → bỏ qua, không nối tiếp")
    _ck = []
    print(f"[TRAIN]   (cũ: {_old_sig} | nay: {_sig})")
try:
    json.dump(_sig, open(_sig_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

_resume = os.path.join(CKPT_DIR, _ck[-1]) if _ck else None
print(f"[TRAIN] {'tiếp tục từ ' + _resume if _resume else 'bắt đầu từ đầu'}")

trainer_stats = trainer.train(resume_from_checkpoint=_resume)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\n[SAVE] adapter → {ADAPTER_DIR}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

logs = pd.DataFrame(trainer.state.log_history)
tr = logs[logs.get("loss").notna()] if "loss" in logs else pd.DataFrame()
ev = logs[logs.get("eval_loss").notna()] if "eval_loss" in logs else pd.DataFrame()

plt.figure(figsize=(9, 5))
if not tr.empty:
    plt.plot(tr["step"], tr["loss"], label="train loss")
if not ev.empty:
    plt.plot(ev["step"], ev["eval_loss"], marker="o", label="val loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title("SFT Qwen3-8B — rejection sampling data")
_p = os.path.join(OUTPUT_DIR, f"sft_loss_{STAMP}.png")
plt.savefig(_p, dpi=150); plt.show()

if not ev.empty:
    _best = ev.loc[ev["eval_loss"].idxmin()]
    print(f"  val loss thấp nhất = {_best['eval_loss']:.4f} tại step {int(_best['step'])}")
    if ev["eval_loss"].iloc[-1] > ev["eval_loss"].min() * 1.05:
        print("  ⚠ val loss đã tăng trở lại → có dấu hiệu overfit; early stopping đã chặn.")
    else:
        print("  ✅ val loss không tăng ngược — khác hẳn đường cong SFT của cách cũ.")
print(f"[SAVE] {_p}")

# Lịch sử huấn luyện: đường cong loss + cấu hình, để soi lại mà không cần chạy lại.
_hist = os.path.join(OUTPUT_DIR, f"sft_history_{STAMP}.json")
json.dump({"stamp": STAMP, "log_history": trainer.state.log_history,
           "best_eval_loss": (float(ev["eval_loss"].min()) if not ev.empty else None),
           "best_step": (int(ev.loc[ev["eval_loss"].idxmin(), "step"])
                         if not ev.empty else None),
           "n_train": len(train_ds), "n_val": len(val_ds),
           "lora": dict(sft.lora_config(),
                        use_gradient_checkpointing=GC_MODE),
           "training_args": cfg, "batch_meta": meta,
           "sft_data": SFT_JSONL,
           "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
           "env": run_env()},
          open(_hist, "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
print(f"[SAVE] {_hist}   ← đường cong loss + cấu hình huấn luyện")

---

# ⚠ RESTART RUNTIME TẠI ĐÂY (lần 2)

**Runtime → Restart session**, rồi chạy **từ cell #17 trở đi** (chính là §10 bên dưới).

Đừng chạy lại §6–§9: §9 là ô huấn luyện, chạy lại là train lại từ đầu mất 28 phút.
Phần C tự lo đủ mọi thứ nó cần — cài đặt, cấu hình, nạp model, nạp adapter.

Lý do restart: phiên hiện tại đang ở chế độ huấn luyện, cần quay lại engine vLLM để sinh.
Adapter đã lưu trên Drive nên không mất gì.

---

# PHẦN C — Chấm model đã SFT

## §10. Chấm trên tập test

Adapter được nạp vào engine vLLM qua `model.load_lora`, prompt giữ nguyên của nấc 2 — đúng
định dạng model vừa được huấn luyện.

In [ ]:
# Cài đặt — ghim theo bộ ĐÃ XÁC MINH cài xong sạch trên image Colab hiện tại
# (Python 3.13, torch 2.11.0+cu128, A100).
#
# ⚠ KHÁC bản tham chiếu, và đây là chủ ý:
#   Khối cài đặt gốc ghim transformers==4.56.2 / trl==0.22.2 / xformers==0.0.29.post3.
#   Trên image Colab hiện tại nó THẤT BẠI — nhánh chọn xformers chỉ biết torch 2.8/2.9,
#   gặp torch 2.11 thì rơi vào bản 0.0.29.post3 (dành cho torch 2.5) nên đổ cả khối,
#   mà `%%capture` lại nuốt mất báo lỗi.
#   Bộ dưới đây là bộ pip tự giải ra khi để `unsloth` và `vllm` thoả thuận với nhau.
#   Chênh lệch phiên bản được ghi vào `env` của meta mỗi nấc, nên báo cáo vẫn truy được.
#
# ⏱ 6–12 phút (đã ghim nên pip khỏi dò tìm). Cố ý KHÔNG giấu output để thấy nó còn sống.
import os, time
_t_cai = time.time()
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install unsloth==2026.9.4 transformers==4.57.6 trl==0.24.0 peft==0.20.0 bitsandbytes==0.50.2 xformers==0.0.35
    # vLLM phải khớp CUDA của torch. Bản trên PyPI dựng cho CUDA 13, còn Colab đang
    # CUDA 12.8 → unsloth CHẶN import và báo "No module named 'vllm'" dù gói vẫn có.
    # Wheel dưới đây là bản cu129, đúng cái unsloth khuyến nghị cho hệ CUDA 12.x.
    # Nếu image Colab đổi CUDA: chạy ô này, đọc dòng WARNING của unsloth ở cell sau —
    # nó in ra đúng URL wheel cần dùng, thay vào đây là xong.
    !pip install https://github.com/vllm-project/vllm/releases/download/v0.23.0/vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl
print(f"\n[CÀI ĐẶT] xong sau {(time.time() - _t_cai) / 60:.1f} phút")
print("[CÀI ĐẶT] Colab hiện nút RESTART SESSION thì bấm, rồi chạy lại TỪ CELL #2 "
      "(bỏ qua ô này — cài lại là thừa).")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, sft, stats
from vinumqa.prompts import PromptKit

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

# Thang prompt LỒNG NHAU: basic ⊂ no_fewshot ⊂ engineered — mỗi nấc thêm đúng một khối
# cắt ra từ prompt hoàn chỉnh, phần chung giống nhau từng ký tự.
LADDER = [
    ("01_basic",           "Nấc 1 — prompt cơ bản (danh sách phép toán + yêu cầu)"),
    ("02_prompt_eng",      "Nấc 2 — prompt hoàn chỉnh (+ hướng dẫn từ khoá + few-shot)"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("04_selfeval_sft",    "Nấc 4b — + self-eval (model SFT)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05_ace_sft",         "Nấc 5b — + ACE (model SFT)"),
    ("05c_ace_basic_base", "Nấc 5c — ACE trên prompt cơ bản"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("06_comb_F_A",        "Tổ hợp — SFT + ACE (không self-eval)"),
    ("08_tu_nhat_quan",    "Mới — self-consistency K mẫu (ví dụ cố định)"),
    ("09_vidu_dong",       "Mới — self-consistency + ví dụ truy hồi"),
    ("06_comb_E_A_moi",    "Mục tiêu — prompt + ACE + phương pháp mới"),
    ("06_comb_F_A_moi",    "Mục tiêu — SFT + ACE + phương pháp mới"),
]
LADDER_LABEL = dict(LADDER)

# Những biến phải đi kèm kết quả thì mới truy lại được về sau.
_CFG_KEYS = ("MODEL_NAME", "MODEL_TAG", "TEMPERATURE", "MAX_TOKENS", "REPETITION_PENALTY",
             "MAX_SEQ_LENGTH", "BATCH_SIZE", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "RANDOM_SEED")


def run_env():
    """Môi trường THẬT lúc chạy: commit, GPU, phiên bản thư viện.

    Chỉ đọc thư viện đã nạp (``sys.modules``) chứ không import thêm — vừa nhanh,
    vừa báo đúng những gì thật sự được dùng.
    """
    env = {"python": sys.version.split()[0]}
    try:                                   # bản code nào sinh ra kết quả này
        def _g(*a):
            return subprocess.run(["git", "-C", REPO_DIR, *a],
                                  capture_output=True, text=True).stdout.strip()
        env["commit"] = _g("rev-parse", "--short", "HEAD")
        env["branch"] = _g("rev-parse", "--abbrev-ref", "HEAD")
        env["dirty"] = bool(_g("status", "--porcelain"))
    except Exception:
        pass
    _torch = sys.modules.get("torch")
    if _torch is not None:
        env["torch"] = getattr(_torch, "__version__", "?")
        try:
            if _torch.cuda.is_available():
                _p = _torch.cuda.get_device_properties(0)
                env["gpu"] = _p.name
                env["vram_gb"] = round(_p.total_memory / 1024**3, 1)
                env["cc"] = f"{_p.major}.{_p.minor}"
            else:
                env["gpu"] = "CPU"
        except Exception:
            pass
    else:
        env["gpu"] = "CPU (không nạp torch)"
    for _lib in ("transformers", "trl", "peft", "vllm", "unsloth",
                 "sentence_transformers", "numpy"):
        _m = sys.modules.get(_lib)
        if _m is not None and hasattr(_m, "__version__"):
            env[_lib] = _m.__version__
    return env


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "meta":  f"{stage}_meta.json"}[kind])


# Cấu hình CHUẨN của cả thang bậc — đo trên A100 40GB.
# MAX_SEQ_LENGTH đổi theo GPU (A100 15000 / L4 13500 / T4 8192), mà đổi GPU là đổi
# thành phần lô, đổi kernel, đổi luôn token được lấy mẫu ở temperature 0.1. Hai lần
# chạy khác max_seq KHÔNG so thẳng được, nên phải ghi sang tên nấc khác.
MAX_TOKENS_CHUAN, MAX_SEQ_CHUAN = 4096, 17000



def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: jsonl + meta, kèm một file output thô.

    Chạy với ``MAX_TOKENS`` khác mức chuẩn thì tự ghi sang tên nấc khác. Đổi trần sinh
    là đổi cấu hình, kết quả không so thẳng với thang bậc được — mà nếu cứ ghi đè lên
    tên cũ thì mất luôn bản chuẩn, không lấy lại được nếu không chạy lại GPU.
    """
    _hau_to = ""
    _mt, _ms = globals().get("MAX_TOKENS"), globals().get("MAX_SEQ_LENGTH")
    if _mt and _mt != MAX_TOKENS_CHUAN:
        _hau_to += f"_tok{_mt}"
    if _ms and _ms != MAX_SEQ_CHUAN:
        _hau_to += f"_seq{_ms}"
    if _hau_to and not stage.endswith(_hau_to):
        stage = f"{stage}{_hau_to}"
        if not quiet:
            print(f"[GHI] ⚠ cấu hình khác chuẩn (max_tokens={_mt}, max_seq={_ms}, "
                  f"GPU={globals().get('_GPU', '?')}) → ghi sang nấc '{stage}'.")
            print( "       Kết quả khác GPU/khác trần KHÔNG so thẳng với thang bậc chuẩn.")
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            "enable_thinking": getattr(globals().get("prompt_kit", None),
                                       "enable_thinking", "?"),
            "ty_le_bi_cat_token": (round(ty_le_bi_cat(), 4)
                                   if "ty_le_bi_cat" in globals() else None),
            "bi_cat_theo_buoc": (bi_cat_theo_buoc()
                                if "bi_cat_theo_buoc" in globals() else None),
            "nap_an_toan": bool(globals().get("NAP_AN_TOAN", False)),
            "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
            "env": run_env(),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'meta')}")
    return stage                      # tên THẬT, có thể khác tên truyền vào


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else "máy cá nhân"
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

In [ ]:
# ═══════════════ MODEL — Qwen3-8B 4-bit ═══════════════
# Tham số lấy từ reference/original_notebooks/inference_with_difference_models.ipynb:
#   load_in_4bit=True, fast_inference=True, temperature=0.1
# max_tokens thì KHÔNG giữ: nâng 3000 → 8192 vì ở mức cũ 5–10 % mẫu bị cắt giữa lúc
# suy nghĩ, mất trắng. Xem lý do đầy đủ ở ô cấu hình GPU.
MODEL_NAME = "unsloth/Qwen3-8B"
MODEL_TAG  = "Qwen3-8B"

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Không thấy GPU. Runtime → Change runtime type → L4 GPU.")
_GPU, _VRAM = torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1024**3
_CC = torch.cuda.get_device_capability(0)
if _CC[0] < 7:
    raise RuntimeError(f"{_GPU} (CC {_CC[0]}.{_CC[1]}) không chạy được vLLM. "
                       f"Runtime → Change runtime type → A100 GPU.")

# ═══ Tham số ẢNH HƯỞNG KẾT QUẢ — CỐ ĐỊNH trên mọi GPU ═══
# Trước đây max_seq đổi theo GPU (A100 15000 / L4 13500) nên hai máy cho kết quả
# không so thẳng được. Giờ khoá cứng: đổi GPU chỉ đổi tốc độ, không đổi đầu vào.
#
# TRẦN SINH = 4096. Đây là mức ĐO ĐƯỢC là tối ưu, không phải chọn bừa:
#     nấc 2, cùng prompt, cùng GPU, chỉ khác trần —
#       4096 → 30 lượt bị cắt | 28 mẫu mất | EA 0.6479 | 300 mẫu đúng
#       8192 → 30 lượt        | 28 mẫu     | EA 0.6479 | 300 mẫu đúng
#     Gấp đôi ngân sách cứu ĐÚNG 0 mẫu. Số mẫu vượt trần không phụ thuộc trần, nên 4096
#     đã qua điểm bão hoà; 8192 chỉ tốn thêm thời gian. (Dưới 4096 thì mất thêm mẫu.)
#
# ~6 % mẫu vẫn chạm trần — nay KHÔNG bỏ mặc nữa: run_pipeline vớt chúng bằng một lượt
# sinh lại với suy nghĩ TẮT (xem `vot_mau_bi_cat`). Đó mới là cách chữa, không phải trần.
#
# max_seq 17000 theo ngân sách (neo vào phép đo thật bằng tokenizer):
#     prompt bước 2 xấu nhất = 7464 + 4096 = 11560
#     ngân sách              = 17000 − 4096 = 12904   → dư 1344 token
# Ô §3 đo lại bằng tokenizer thật và tự cắt ngữ cảnh + báo động nếu tính sai.
#
# ⚠ ĐỪNG nâng tiếp. Đã có phép so SẠCH: nấc 2 chạy hai lần với CÙNG prompt engineered,
# cùng model, cùng GPU, chỉ khác trần token —
#     trần 4096 → 30 lượt sinh bị cắt | 28 mẫu mất trắng | EA 0.6479 | 300 mẫu đúng
#     trần 8192 → 30 lượt             | 28 mẫu           | EA 0.6479 | 300 mẫu đúng
# Gấp đôi ngân sách cứu được ĐÚNG 0 mẫu, đổi lại ~50 % thời gian (10,9 → 16,4 phút).
#
# Số mẫu vượt ngân sách KHÔNG phụ thuộc ngân sách → những lượt đó thực tế không có điểm
# dừng. Mà chúng cũng không lặp (§4 đo trung vị lặp = 0.0 ở nấc 2), nên repetition_penalty
# cũng không phải thuốc. Coi đây là sàn ~6 %, đều ở mọi nấc: ghi nhận rồi bỏ qua.
TEMPERATURE, MAX_TOKENS = 0.1, 4096
MAX_SEQ_LENGTH = 17000
REPETITION_PENALTY = 1.0

# ═══ Tham số chỉ ảnh hưởng TỐC ĐỘ — chỉnh theo VRAM ═══
if _VRAM < 20:
    raise RuntimeError(
        f"{_GPU} chỉ {_VRAM:.0f} GB — không đủ cho max_seq={MAX_SEQ_LENGTH}.\n"
        f"Hạ max_seq xuống thì kết quả KHÔNG so được với các nấc khác, nên thà dừng "
        f"còn hơn ra một con số không dùng được. Đổi sang L4 hoặc A100.")
# util giữ 0.85 (hạ từ 0.88 sau một lần vLLM không dựng nổi engine vì VRAM còn sót).
# MAX_NUM_SEQS trả về mức cũ được vì max_seq đã từ 25000 xuống 17000, áp lực KV giảm hẳn.
#
# BATCH_SIZE = 512 để 497 mẫu vào ĐÚNG MỘT LÔ. Đo từ log thật: lô 400 mẫu chạy
# 1,88 s/mẫu, lô 97 mẫu còn lại chạy 2,83 s/mẫu — chậm hơn 50 % vì không lấp đầy GPU mà
# vẫn phải đợi mẫu dài nhất. Gộp một lô tiết kiệm ~1,5 phút MỖI lượt sinh; nấc 4 và nấc 5
# có nhiều lượt nên cộng lại đáng kể.
elif _VRAM < 30:                           # L4 24GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.86, 16, 512
elif _VRAM < 60:                           # A100 40GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.85, 48, 512
else:                                      # A100 80GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.85, 128, 512
DTYPE = torch.float16 if _CC[0] < 8 else None

print(f"[GPU] {_GPU} | {_VRAM:.1f} GB | CC {_CC[0]}.{_CC[1]}")
print(f"[CFG] max_seq={MAX_SEQ_LENGTH} max_tokens={MAX_TOKENS} temp={TEMPERATURE} "
      f"(cố định mọi GPU) | batch={BATCH_SIZE} max_num_seqs={MAX_NUM_SEQS} (theo VRAM)")

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import unsloth
from unsloth import FastLanguageModel
from vllm import SamplingParams

torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)

import shutil as _sh

# Đặt True nếu model tải về bị thiếu trọng số: tắt hf_transfer thì tải chậm hơn vài phút
# nhưng có kiểm tra và tải tiếp được. Lưu ý: `export` trong Cửa sổ dòng lệnh KHÔNG tới
# được kernel notebook — phải đặt ở đây.
TAI_CHAM_CHO_CHAC = False
if TAI_CHAM_CHO_CHAC:
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    print("[MODEL] đã tắt hf_transfer — tải chậm hơn nhưng chắc hơn")

_free = _sh.disk_usage("/").free / 1024**3
_t_nap = time.time()
print(f"[MODEL] Đang tải {MODEL_NAME} ... (đĩa trống {_free:.0f} GB)")
if _free < 15:
    print("[MODEL] ⚠ dưới 15 GB trống — model ~15 GB, tải dễ đứt giữa chừng.")
print("[MODEL] ⏳ Mất 4–7 PHÚT. Tải xong rồi vLLM còn dựng CUDA graph — đoạn đó")
print("[MODEL]    KHÔNG có thanh tiến trình, nhìn như treo nhưng không phải.")
print("[MODEL]    Muốn biết còn sống: xem MỐC GIỜ ở các dòng INFO bên dưới. Nó nhích")
print("[MODEL]    lên là đang chạy. Đứng im quá 10 phút mới đáng nghi.")

# enable_prefix_caching: system prompt (~1 800 token) GIỐNG HỆT ở cả 497 request, nên
# vLLM chỉ cần prefill nó một lần rồi dùng lại. Tiết kiệm phần lớn thời gian prefill.
# Không đổi token sinh ra — mỗi request vẫn có seed riêng.
_NAP_KW = dict(model_name     = MODEL_NAME,
               dtype          = DTYPE,
               max_seq_length = MAX_SEQ_LENGTH,
               load_in_4bit   = True,
               fast_inference = True)
try:                                  # bản unsloth cũ không nhận tham số này
    import inspect as _insp
    if "enable_prefix_caching" in _insp.signature(
            FastLanguageModel.from_pretrained).parameters:
        _NAP_KW["enable_prefix_caching"] = True
except Exception:                                    # noqa: BLE001
    pass
NAP_AN_TOAN = False          # True = đã phải lùi về chế độ an toàn, có ghi vào meta

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        **_NAP_KW, gpu_memory_utilization=GPU_MEM_UTIL, max_num_seqs=MAX_NUM_SEQS)
except (RuntimeError, ValueError) as _e:
    # ── vLLM dựng engine hỏng vì CUDA ──
    # KHÔNG phải tải model hỏng: model đã nằm trên đĩa rồi. Lỗi ở bước cấp phát KV cache
    # và dựng CUDA graph — thường do VRAM trống ít hơn lần trước (GPU khác, hoặc tiến
    # trình cũ còn giữ bộ nhớ), khiến số block KV tính ra quá nhỏ.
    if "CUDA error" in str(_e) or "invalid argument" in str(_e):
        print("[MODEL] ⚠ vLLM KHÔNG dựng được engine (CUDA error).")
        print(f"[MODEL]   Đang dùng: max_seq={MAX_SEQ_LENGTH} util={GPU_MEM_UTIL} "
              f"max_num_seqs={MAX_NUM_SEQS}")
        try:
            _free, _tot = torch.cuda.mem_get_info()
            print(f"[MODEL]   VRAM trống: {_free/1024**3:.1f}/{_tot/1024**3:.1f} GB"
                  + ("   ← ĐÃ BỊ CHIẾM. Restart session rồi chạy lại TỪ Ô #2."
                     if _free / _tot < 0.9 else ""))
        except Exception:                                    # noqa: BLE001
            pass
        print("[MODEL]   Thử lại ở CHẾ ĐỘ AN TOÀN: bỏ CUDA graph, hạ VRAM và số chuỗi.")
        print("[MODEL]   Ba thứ đó chỉ đổi TỐC ĐỘ — mỗi request đã có seed riêng nên")
        print("[MODEL]   thành phần lô không ảnh hưởng token sinh ra.")
        gc.collect()
        torch.cuda.empty_cache()
        _an = dict(gpu_memory_utilization=min(GPU_MEM_UTIL, 0.80),
                   max_num_seqs=max(8, MAX_NUM_SEQS // 4))
        try:
            model, tokenizer = FastLanguageModel.from_pretrained(
                **_NAP_KW, enforce_eager=True, **_an)
        except TypeError:                 # bản unsloth không nhận enforce_eager
            model, tokenizer = FastLanguageModel.from_pretrained(**_NAP_KW, **_an)
        GPU_MEM_UTIL = _an["gpu_memory_utilization"]
        MAX_NUM_SEQS = _an["max_num_seqs"]
        NAP_AN_TOAN = True
        print(f"[MODEL] ✅ nạp được ở chế độ an toàn (util={GPU_MEM_UTIL} "
              f"max_num_seqs={MAX_NUM_SEQS}) — chậm hơn, kết quả không đổi.")
    # ── Thiếu trọng số: shard safetensors tải dở còn trong cache ──
    elif "not initialized from checkpoint" in str(_e):
        raise RuntimeError(
            "Model thiếu trọng số — bản tải dở trong cache HuggingFace.\n\n"
            "Bước 1 — xoá cache. Mở Cửa sổ dòng lệnh (góc dưới trái):\n"
            "    rm -rf ~/.cache/huggingface/hub/models--unsloth--Qwen3-8B*\n"
            "    df -h / | tail -1          # kiểm luôn, cần ≥ 20 GB trống\n\n"
            "Bước 2 — đặt TAI_CHAM_CHO_CHAC = True ở ĐẦU CHÍNH Ô NÀY.\n"
            "    (`export` trong terminal không tới được kernel notebook.)\n\n"
            "Bước 3 — Restart session, chạy lại TỪ CELL #2 (bỏ qua ô cài đặt).\n\n"
            "Hỏng y hệt lần nữa thì không phải do tải: khi đó là bản 4-bit của unsloth "
            "không khớp bộ nạp của vLLM, phải đổi phiên bản chứ không phải tải lại.") from _e
    else:
        raise

SAMPLING = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                          repetition_penalty=REPETITION_PENALTY, seed=RANDOM_SEED)
LORA_REQUEST = None          # nấc 3 trở đi có thể gán adapter đã SFT vào đây

from collections import Counter as _Counter
LY_DO_DUNG = _Counter()          # finish_reason: "stop" = tự kết thúc, "length" = BỊ CẮT
LY_DO_THEO_BUOC = {}             # desc → Counter riêng, để tách bước 1 với bước 2


def ty_le_bi_cat():
    """Phần trăm lượt sinh bị cắt vì chạm max_tokens, TÍNH TỪ LẦN reset gần nhất."""
    t = sum(LY_DO_DUNG.values())
    return (LY_DO_DUNG.get("length", 0) / t) if t else 0.0


def bi_cat_theo_buoc():
    """Tỉ lệ bị cắt TÁCH RIÊNG cho bước 1 và bước 2.

    Phải tách vì prompt bước 2 (self-eval ở nấc 4, ACE ở nấc 5) chứa NGUYÊN lời giải
    bước 1, nên dài hơn bước 1 rất nhiều. Bước 2 bị cắt nhiều hơn nghĩa là phương pháp
    bị PHA LOÃNG — mất cơ hội sửa, chứ không phải sửa sai. Con số gộp chung không phân
    biệt được hai chuyện đó.

    Gom theo đuôi của desc ("vòng3/step1" và "vòng7/step1" cùng vào "step1").
    """
    gom = {}
    for k, c in LY_DO_THEO_BUOC.items():
        gom.setdefault(k.rsplit("/", 1)[-1], _Counter()).update(c)
    return {b: {"n": sum(c.values()), "bi_cat": c.get("length", 0),
                "ty_le": round(c.get("length", 0) / max(1, sum(c.values())), 4)}
            for b, c in sorted(gom.items())}


def in_bi_cat_theo_buoc():
    d = bi_cat_theo_buoc()
    if not d:
        return
    # In cả khi chỉ có MỘT bước: nấc 1 và 2 cũng cần biết tỉ lệ chạm trần của mình,
    # nếu không thì mãi tới nấc 4 mới thấy con số đó.
    print("   Bị cắt vì trần token, tách theo bước:")
    for b, v in d.items():
        print(f"     {b:<10}{v['ty_le']:>7.1%}  ({v['bi_cat']}/{v['n']} lượt)")
    if "step2" in d and "step1" in d and d["step2"]["ty_le"] > d["step1"]["ty_le"] + 0.02:
        print("     ⚠ bước 2 bị cắt nhiều hơn bước 1 → hiệu quả của phương pháp đang bị")
        print("       PHA LOÃNG (mất cơ hội sửa). Hiệu số đo được là cận DƯỚI.")


def dat_lai_bo_dem():
    """Gọi ngay trước mỗi nấc. Không gọi thì tỉ lệ là cộng dồn cả phiên — gồm cả
    lượt warmup và (ở nấc 5) toàn bộ pha A, không phản ánh nấc đang đo."""
    LY_DO_DUNG.clear()
    LY_DO_THEO_BUOC.clear()


def generate(prompts, sampling_params=None, desc=None, batch_size=None):
    """Sinh theo lô qua vLLM — như vòng lặp trong notebook cũ."""
    if not prompts:
        return []
    sp = sampling_params or SAMPLING
    bs = batch_size or BATCH_SIZE
    outs, t0 = [], time.time()
    nb = (len(prompts) + bs - 1) // bs
    for i in range(nb):
        chunk = prompts[i*bs:(i+1)*bs]
        kw = {"sampling_params": sp}
        if LORA_REQUEST is not None:
            kw["lora_request"] = LORA_REQUEST
        _res = model.fast_generate(chunk, **kw)
        for o in _res:                      # đếm lý do dừng để biết có bị cắt không
            for _x in o.outputs:            # sinh nhiều mẫu thì đếm CẢ K mẫu
                _r = getattr(_x, "finish_reason", "?")
                LY_DO_DUNG[_r] += 1
                if desc:
                    LY_DO_THEO_BUOC.setdefault(desc, _Counter())[_r] += 1
        # 1 mẫu → trả chuỗi (y như cũ); nhiều mẫu → trả list[str] cho self-consistency.
        outs.extend((o.outputs[0].text if len(o.outputs) == 1
                     else [_x.text for _x in o.outputs]) for o in _res)
        if desc:
            el = time.time() - t0
            print(f"    {desc}: lô {i+1}/{nb} | {el:.0f}s | "
                  f"ETA {el/(i+1)*(nb-i-1):.0f}s", end="\r")
    gc.collect(); torch.cuda.empty_cache()
    if desc:
        _b = LY_DO_THEO_BUOC.get(desc, _Counter())
        _c, _n = _b.get("length", 0), max(1, sum(_b.values()))
        print(f"    {desc}: xong {len(prompts)} prompt trong {time.time()-t0:.0f}s"
              f" | bị cắt vì trần token: {_c/_n:.1%} ({_c} lượt)" + " "*8)
    return outs

_t = torch.cuda.get_device_properties(0).total_memory/1024**3
print(f"[MODEL] ✅ sẵn sàng sau {(time.time()-_t_nap)/60:.1f} phút | "
      f"VRAM {_t - torch.cuda.mem_get_info()[0]/1024**3:.1f}/{_t:.1f} GB")
_ = generate(["xin chào"], SamplingParams(temperature=0, max_tokens=4))
print("[WARMUP] ✅")

In [ ]:
prompt_kit = PromptKit(tokenizer=tokenizer, model_name=MODEL_NAME)
PROMPT_LEVEL = "engineered"
USE_SELFEVAL = False

_think = getattr(prompt_kit, "enable_thinking", None)
print(f"[PROMPT] mức = {PROMPT_LEVEL} | self-eval = {USE_SELFEVAL} | "
      f"suy nghĩ = {'template tự quyết (Qwen3: BẬT)' if _think is None else _think}")
if _think is False:
    print("[PROMPT] ⚠ suy nghĩ đang TẮT — lệch bản tham chiếu, PA sẽ hụt "
          "~10 điểm. Dấu hiệu: 497 mẫu chạy xong trong ~1 phút.")
print(f"         thang lồng nhau: basic={len(prompt_kit.BASIC_SYSTEM_PROMPT)} ký tự"
      f" ⊂ no_fewshot={len(prompt_kit.NO_FEWSHOT_SYSTEM_PROMPT)}"
      f" ⊂ engineered={len(prompt_kit.ENGINEERED_SYSTEM_PROMPT)}"
      f" | self-eval={len(prompt_kit.SELF_EVAL_SYSTEM_PROMPT)}")

# Đo bằng tokenizer THẬT trên 40 mẫu có ngữ cảnh DÀI NHẤT
BUDGET = MAX_SEQ_LENGTH - MAX_TOKENS
_clen = lambda s: (len(" ".join(s.get("pre_text") or [])) +
                   len(" ".join(s.get("post_text") or [])) + len(str(s.get("table") or "")))
_probe = sorted(test_all, key=_clen, reverse=True)[:40]
_bul = "\n".join(["- Khi hỏi tốc độ tăng trưởng, dùng subtract(gia_tri_moi, gia_tri_cu), "
                  "divide(#0, gia_tri_cu)."] * 7)
# Lời giải bước 1 dài nhất có thể là đúng MAX_TOKENS token (model sinh chạm trần).
# Phải đo ở mức đó, không thì bật suy nghĩ vào là prompt bước 2 tràn ngân sách.
_unit = "Phân tích chi tiết từng bước của bảng số liệu. "
_prev = _unit * max(1, MAX_TOKENS // max(1, len(tokenizer(_unit).input_ids)))
_prev += "\n```plaintext\nprogram: divide(1,2)\nanswer: 0.5\n```"

def _measure():
    a = [len(tokenizer(prompt_kit.step1(s, _bul, level=PROMPT_LEVEL)).input_ids)
         for s in _probe]
    b = ([len(tokenizer(prompt_kit.step2(s, _prev, _bul)).input_ids) for s in _probe]
         if USE_SELFEVAL else [0])
    return a, b

_a, _b = _measure()
print(f"[PROMPT] (40 mẫu dài nhất) step1 max={max(_a)} | step2 max={max(_b)} | "
      f"ngân sách={BUDGET}")

if max(max(_a), max(_b)) > BUDGET:
    prompt_kit.max_prev_chars = 3000
    _cap = _clen(_probe[0])
    for _ in range(6):
        _cap = int(_cap * 0.80)
        prompt_kit.max_ctx_chars = max(1200, _cap)
        _a, _b = _measure()
        if max(max(_a), max(_b)) <= BUDGET:
            break
    assert max(max(_a), max(_b)) <= BUDGET, "Không cắt đủ — giảm MAX_TOKENS hoặc dùng GPU lớn hơn."
    _hit = sum(1 for s in test_all if _clen(s) > prompt_kit.max_ctx_chars)
    print(f"[PROMPT] ⚠ đã bật cắt ngữ cảnh (max_ctx_chars={prompt_kit.max_ctx_chars}); "
          f"{_hit}/{len(test_all)} mẫu bị cắt ({_hit/len(test_all)*100:.1f}%)")
    print(f"[PROMPT]   GHI LẠI con số này khi báo cáo.")
else:
    print("[PROMPT] ✅ mọi prompt đều lọt ngân sách, không cần cắt")

In [ ]:
# ═══════════════ NẠP ADAPTER SFT VÀO vLLM ═══════════════
import os
import json

ADAPTER_DIR = os.path.join(OUTPUT_DIR, "sft_adapter_qwen3")
assert os.path.isdir(ADAPTER_DIR), (
    f"Không thấy adapter tại {ADAPTER_DIR} — chạy §9 trước."
)

# ── Kiểm tra adapter có đủ file cần thiết ──────────────────────────────────
_adapter_cfg = os.path.join(ADAPTER_DIR, "adapter_config.json")
assert os.path.isfile(_adapter_cfg), (
    f"Không thấy adapter_config.json trong {ADAPTER_DIR}"
)

_adapter_weights = [
    os.path.join(ADAPTER_DIR, "adapter_model.safetensors"),
    os.path.join(ADAPTER_DIR, "adapter_model.bin"),
]
assert any(os.path.isfile(p) for p in _adapter_weights), (
    f"Không thấy adapter_model.safetensors/bin trong {ADAPTER_DIR}"
)

with open(_adapter_cfg, encoding="utf-8") as f:
    _lora_cfg = json.load(f)

print(
    f"[LoRA] adapter rank={_lora_cfg.get('r', '?')} | "
    f"alpha={_lora_cfg.get('lora_alpha', '?')}"
)


# ── Đọc lại con trỏ dữ liệu SFT ────────────────────────────────────────────
_latest = os.path.join(OUTPUT_DIR, "sft_data", "latest.txt")

SFT_JSONL = (
    open(_latest, encoding="utf-8").read().strip()
    if os.path.exists(_latest)
    else None
)

N_SFT_RECORDS = (
    sum(1 for _ in open(SFT_JSONL, encoding="utf-8"))
    if SFT_JSONL and os.path.exists(SFT_JSONL)
    else None
)

print(f"[SFT] huấn luyện từ: {SFT_JSONL} ({N_SFT_RECORDS} mẫu)")


# ── Model ở trên PHẢI là model fast_inference=True ─────────────────────────
assert hasattr(model, "fast_generate"), (
    "model không có fast_generate — model phải được nạp với fast_inference=True."
)

assert hasattr(model, "vllm_engine"), (
    "Không thấy vLLM engine. Hãy chạy lại cell MODEL với fast_inference=True."
)


# ── Nạp LoRA cho vLLM ──────────────────────────────────────────────────────
# Đường lùi nằm trong `sft.nap_lora` để 04/05/06 dùng chung một bản, khỏi trôi lệch.
LORA_REQUEST = sft.nap_lora(model, ADAPTER_DIR)
_lora_loader = ("model.load_lora" if hasattr(model, "load_lora")
                else "unsloth_zoo.vllm_utils.load_lora")

print(f"[LoRA] ✅ đã nạp adapter từ {ADAPTER_DIR}")
print(f"[LoRA] loader = {_lora_loader}")
print(f"[LoRA] request = {LORA_REQUEST}")
print("       generate() sẽ tự truyền lora_request vào model.fast_generate().")

In [ ]:
dat_lai_bo_dem()          # tỉ lệ bị cắt tính riêng cho nấc này
STAGE = "03_sft"
print(f"\n{'═'*74}\n  NẤC: {STAGE} | prompt={PROMPT_LEVEL} | self-eval={USE_SELFEVAL}"
      f" | {len(test_all)} mẫu\n{'═'*74}")

_t0 = time.time()
rows = pipeline.run_pipeline(
    test_all, prompt_kit, generate,
    prompt_level=PROMPT_LEVEL, use_selfeval=USE_SELFEVAL,
    sp_step1=SAMPLING, sp_step2=SAMPLING, desc=STAGE)
metrics = pipeline.summarize(rows, STAGE)
metrics["minutes"] = round((time.time() - _t0) / 60, 1)
pipeline.print_summary(metrics)
in_bi_cat_theo_buoc()
print(f"\n  Thời gian: {metrics['minutes']} phút")

### So với nấc 2

In [ ]:
_prev_rows = load_stage("02_prompt_eng")
if _prev_rows is None:
    print("[SO SÁNH] ⚠ chưa có kết quả nấc '02_prompt_eng' → bỏ qua phần kiểm định.")
    print("          Chạy notebook nấc trước rồi quay lại cell này.")
else:
    _m_prev = pipeline.summarize(_prev_rows, "02_prompt_eng")
    print(f"\n{'═'*84}\n  NẤC 3 vs NẤC 2 — giá trị của SFT\n{'═'*84}")
    print(f"{'nấc':<28}{'EA':>10}{'PA_strict':>12}{'PA_loose':>11}{'no_prog':>10}")
    for _n, _m in [("02_prompt_eng", _m_prev), ("03_sft", metrics)]:
        print(f"{_n:<28}{_m['EA']:>10.4f}{_m['PA_strict']:>12.4f}"
              f"{_m['PA_loose']:>11.4f}{_m['no_program']:>10.4f}")
    print(f"\n  Δ EA        = {metrics['EA'] - _m_prev['EA']:+.4f}")
    print(f"  Δ PA_strict = {metrics['PA_strict'] - _m_prev['PA_strict']:+.4f}")

    for _k in ("ea", "pa_strict"):
        stats.compare_pair(_prev_rows, rows, key=_k,
                           label="NẤC 3 vs NẤC 2 — giá trị của SFT",
                           name_base="02_prompt_eng", name_variant="03_sft")

In [ ]:
STAGE_DA_GHI = save_stage("03_sft", rows, metrics,
           extra={"prompt_level": "engineered", "self_eval": False,
                  "adapter": ADAPTER_DIR,
                  "sft_data": SFT_JSONL if "SFT_JSONL" in dir() else None,
                  "n_sft_records": (N_SFT_RECORDS if "N_SFT_RECORDS" in dir()
                                    else len(records) if "records" in dir() else None)})
print(f"\n  EA = {metrics['EA']:.4f} | PA_strict = {metrics['PA_strict']:.4f}")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  KHỐI ĐỂ GỬI ĐI ĐỐI CHIẾU — bôi đen từ dòng ─── tới hết rồi copy         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
print("\n" + "─" * 74)
print(json.dumps(json.load(open(stage_path(STAGE_DA_GHI, "meta"), encoding="utf-8")),
                 ensure_ascii=False, indent=1))
print("─" * 74)


## Kết luận nấc 3

Ba khả năng, và mỗi khả năng có một kết luận khác nhau cho báo cáo:

| Kết quả | Nghĩa là | Viết gì trong bài |
|---|---|---|
| SFT > nấc 2, p < 0.05 | Rejection sampling khắc phục được thất bại SFT của cách cũ | Đóng góp rõ: học được từ train mà không overfit, **không cần API ngoài** |
| SFT ≈ nấc 2 | Model đã bão hoà với dữ liệu tự sinh | Vẫn đáng báo cáo: củng cố luận điểm "inference-time thắng fine-tuning" |
| SFT < nấc 2 | Tự chưng cất làm hẹp phân bố đầu ra | Kiểm tra đường cong loss ở §9; thử `ACCEPT="pa"` cho dữ liệu sạch hơn |

Dù kết quả nào, **nấc 4 và nấc 5 vẫn chạy được trên cả hai model** (gốc và SFT) — hai
notebook sau có cờ `USE_SFT_ADAPTER` để chọn.

**Tiếp theo:** `04_self_evaluation.ipynb`.